# AAROHAN — Human-in-the-Loop Aerial Search & Rescue Review

**SEVA FIRST Innovation Challenge 2026 · INDRA NAT-002 · TRL 2 → 3 software prototype · macOS Apple Silicon (MPS)**

Post-flight review of recorded drone footage:

`OpenCV sampling → YOLO person detection (COCO class 0, MPS) → human reviewer confirms / rejects / flags → audit-trail CSV → precision / recall / FPR / speed`

---

## Pipeline map (run in this order)

| Step | Cell | Purpose |
|------|------|---------|
| **0** | Setup | Environment, MPS verification, single `CFG` object |
| **1** | Sampling | Frame extraction engine (video **or** image folder) |
| **2A** | Data | Choose real data: SARD / HERIDAL / WiSARD / own footage |
| **2B** | HERIDAL test | Optional: 101 labelled test images (~0.5 GB) for GT benchmark |
| **3** | Detect | MPS YOLO + tiled inference for tiny persons; candidate + audit frames |
| **4** | Review | Dashboard with keys `c` `r` `f` `n` `p` `u` |
| **5** | Audit | `detections_log.csv` + append-only `review_events.csv` |
| **6** | Metrics | Precision / recall / FPR / speed; GT comparison when labels exist |
| **7** | Fine-tune | Optional: train on HERIDAL tiles (MPS) and compare with baseline |

**Minimum path:** Setup → Sampling → Data → Detect → Review → Audit → Metrics  
**Benchmark path (no manual review):** Setup → Sampling → HERIDAL test → Detect → Metrics (GT-only)  
**Fine-tune path:** enable flags in Cell 7 one stage at a time.

---

## Design principles (do not break)

1. **Human is final authority** — model only prioritises; every decision is logged.
2. **Streaming memory** — never load a full video into RAM.
3. **Single config** — edit only the USER SETTINGS block in `CFG`.
4. **Honest metrics** — audit sampling estimates recall; Wilson intervals reported.
5. **Reproducible runs** — each run writes `runs/<run_id>/` with config + artefacts.


---
## Cell 1 — Environment, Apple Silicon (MPS), configuration

**What this cell does**
1. Sets `PYTORCH_ENABLE_MPS_FALLBACK` *before* torch import.
2. Installs missing packages into this kernel.
3. Verifies native arm64 + Metal; runs a GPU smoke test.
4. Defines the single `CFG` object used by every later cell.

> If torch was imported earlier in this kernel, **restart the kernel** so the MPS fallback env var takes effect.


In [ ]:
# ==============================================================================
# AAROHAN | CELL 1 — Environment Setup, Apple Silicon (MPS) Verification, Imports
# ==============================================================================
# What this cell does
#   1. Sets PYTORCH_ENABLE_MPS_FALLBACK *before* torch is imported (order matters).
#   2. Installs any missing packages into THIS kernel's Python (no manual pip needed).
#   3. Verifies native arm64 + Metal (MPS) and runs a tiny GPU smoke test.
#   4. Defines ONE configuration object (CFG) that drives every later cell.
#
# Manual install (equivalent) — run once in Terminal inside your venv/conda env:
#   python3 -m pip install -U torch torchvision ultralytics opencv-python \
#       numpy pandas pillow matplotlib ipywidgets tqdm remotezip
# If torch was already imported in this kernel before running this cell,
# restart the kernel so the MPS fallback variable takes effect.
# ------------------------------------------------------------------------------
import os

# Any op Metal does not implement silently runs on CPU instead of crashing.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import importlib.util
import platform
import subprocess
import sys

# import-name -> pip-name
REQUIRED_PACKAGES = {
    "torch": "torch",
    "torchvision": "torchvision",
    "ultralytics": "ultralytics",
    "cv2": "opencv-python",
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "pillow",
    "matplotlib": "matplotlib",
    "ipywidgets": "ipywidgets",
    "tqdm": "tqdm",
    "IPython": "ipython",
    "remotezip": "remotezip",       # lets Cell 2C fetch single files out of a huge remote zip
}


def ensure_packages() -> None:
    """Install whatever is missing into the running interpreter."""
    missing = [pip_name for mod, pip_name in REQUIRED_PACKAGES.items()
               if importlib.util.find_spec(mod) is None]
    if missing:
        print(f"Installing missing packages: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
        importlib.invalidate_caches()


ensure_packages()

# ----------------------------------------------------------------------------
# Imports (everything the later cells need lives here, once)
# ----------------------------------------------------------------------------
import csv
import gc
import getpass
import html
import io
import json
import logging
import math
import random
import re
import time
from collections import OrderedDict
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Callable, Dict, Iterator, List, Optional, Tuple

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
from IPython.display import HTML, Javascript, display
from PIL import Image, ImageDraw
from tqdm.auto import tqdm
from ultralytics import YOLO

try:  # inline plots inside Jupyter (NameError when run as a plain script)
    get_ipython().run_line_magic("matplotlib", "inline")  # noqa: F821
except NameError:
    pass

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-7s | %(message)s",
                    datefmt="%H:%M:%S", force=True)
log = logging.getLogger("aarohan")

# OpenCL on macOS is deprecated and only adds overhead for plain cv2 decode/resize.
try:
    cv2.ocl.setUseOpenCL(False)
except Exception:  # pragma: no cover
    pass


# ----------------------------------------------------------------------------
# Apple Silicon verification
# ----------------------------------------------------------------------------
def _sysctl(name: str) -> Optional[str]:
    """Read a macOS sysctl value (returns None on non-macOS / failure)."""
    try:
        return subprocess.check_output(["sysctl", "-n", name], text=True, timeout=5).strip()
    except Exception:
        return None


def select_device(prefer: str = "mps") -> str:
    """Return 'mps' only if Metal is usable AND passes a real GPU smoke test."""
    if prefer != "mps":
        return "cpu"
    if not torch.backends.mps.is_built():
        print("⚠ This PyTorch build has no MPS support (install the official macOS arm64 wheel).")
        return "cpu"
    if not torch.backends.mps.is_available():
        print("⚠ MPS not available (needs macOS >= 12.3 and an Apple-silicon GPU). Using CPU.")
        return "cpu"
    try:
        x = torch.randn(256, 256, device="mps")
        _ = float((x @ x).sum().cpu())          # forces the kernel to actually run
        torch.mps.synchronize()
        return "mps"
    except Exception as exc:  # driver / OS quirks
        print(f"⚠ MPS smoke test failed ({exc}). Using CPU.")
        return "cpu"


def print_system_report(device: str) -> None:
    ram_bytes = _sysctl("hw.memsize")
    ram_gb = f"{int(ram_bytes) / 1024 ** 3:.0f} GB unified memory" if ram_bytes else "n/a"
    print("=" * 72)
    print(f"Python      : {platform.python_version()}  ({platform.machine()})")
    print(f"macOS       : {platform.mac_ver()[0] or platform.system()}")
    print(f"Chip        : {_sysctl('machdep.cpu.brand_string') or platform.processor() or 'n/a'}  |  {ram_gb}")
    print(f"PyTorch     : {torch.__version__}  | Ultralytics {ultralytics.__version__} | OpenCV {cv2.__version__}")
    print(f"MPS built   : {torch.backends.mps.is_built()}  | MPS available: {torch.backends.mps.is_available()}")
    print(f"Fallback env: PYTORCH_ENABLE_MPS_FALLBACK={os.environ.get('PYTORCH_ENABLE_MPS_FALLBACK')}")
    print(f"ACTIVE DEVICE → {device.upper()}")
    print("=" * 72)
    if platform.system() == "Darwin" and platform.machine() != "arm64":
        print("⚠ Python is running under Rosetta (x86_64). Install a native arm64 Python "
              "(e.g. Miniforge / python.org universal2) or MPS will be unavailable.")


DEVICE = select_device("mps")
print_system_report(DEVICE)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


# ----------------------------------------------------------------------------
# Single configuration object — edit the USER SETTINGS block only
# ----------------------------------------------------------------------------
@dataclass
class AarohanConfig:
    # ---------------- USER SETTINGS: INPUT -----------------------------------
    project_dir: Path = Path.home() / "AAROHAN"
    source_mode: str = "video"            # "video" (drone footage) or "images" (dataset folder)
    video_path: Path = Path.home() / "AAROHAN" / "videos" / "incident_flight.mp4"
    images_dir: Path = Path.home() / "AAROHAN" / "datasets" / "SARD" / "images"
    labels_dir: Optional[Path] = None     # optional ground truth (YOLO .txt or Pascal-VOC .xml)
    data_source_label: str = "real incident footage (no ground truth)"   # logged in the audit trail
    # ---------------- USER SETTINGS: SAMPLING --------------------------------
    sample_interval_s: float = 1.0        # one frame every N seconds (video mode)
    start_s: float = 0.0
    end_s: Optional[float] = None
    max_frames: Optional[int] = None
    fallback_fps: float = 30.0            # used only if the container reports fps <= 0
    jpeg_quality: int = 92
    # ---------------- USER SETTINGS: DETECTOR --------------------------------
    weights: str = "yolo11s.pt"           # any Ultralytics v8/v11 weights (auto-download)
    device: str = DEVICE                  # "mps" or "cpu"
    imgsz: int = 1280                     # full-frame inference size
    conf: float = 0.25                    # confidence threshold
    iou: float = 0.45                     # NMS IoU inside each inference pass
    max_det: int = 300
    agnostic_nms: bool = False
    classes: Tuple[int, ...] = (0,)       # COCO class 0 = person
    # ---------------- USER SETTINGS: TILING (tiny aerial persons) ------------
    use_tiling: bool = True
    tile_size: int = 640
    tile_overlap: float = 0.25
    tile_imgsz: int = 640                 # raise to 960/1280 for very small persons (slower)
    tile_batch: int = 8
    merge_iou: float = 0.50               # cross-tile NMS IoU
    merge_ios: float = 0.85               # cross-tile containment merge (>1 disables)
    # ---------------- USER SETTINGS: BOX FILTERS -----------------------------
    min_box_px: int = 6                   # drop boxes smaller than this (noise)
    max_box_frac: float = 0.25            # drop boxes larger than 25 % of frame (not a person from the air)
    # ---------------- USER SETTINGS: REVIEW QUEUE ----------------------------
    min_detections: int = 1               # frame becomes a candidate at >= N detections
    audit_fraction: float = 0.10          # share of detection-free frames sent for audit (recall estimate)
    display_max_px: int = 1100
    seed: int = 42
    run_id: str = field(default_factory=lambda: datetime.now().strftime("%Y%m%d_%H%M%S"))

    def __post_init__(self):
        for name in ("project_dir", "video_path", "images_dir"):
            setattr(self, name, Path(getattr(self, name)).expanduser())
        if self.labels_dir is not None:
            self.labels_dir = Path(self.labels_dir).expanduser()
        self.classes = tuple(self.classes)
        self.run_dir.mkdir(parents=True, exist_ok=True)
        self.frames_dir.mkdir(parents=True, exist_ok=True)
        (self.project_dir / "videos").mkdir(parents=True, exist_ok=True)
        (self.project_dir / "datasets").mkdir(parents=True, exist_ok=True)

    @property
    def run_dir(self) -> Path:
        return self.project_dir / "runs" / self.run_id

    @property
    def frames_dir(self) -> Path:
        return self.run_dir / "frames"

    @property
    def log_csv(self) -> Path:
        return self.run_dir / "detections_log.csv"

    @property
    def events_csv(self) -> Path:
        return self.run_dir / "review_events.csv"


CFG = AarohanConfig()
print(f"Project dir : {CFG.project_dir}")
print(f"Run id      : {CFG.run_id}   (outputs → {CFG.run_dir})")
print("Edit CFG.video_path / CFG.source_mode, then run Cell 2 onwards.")


---
## Cell 2 — Frame extraction & sampling engine

**Two input modes, one interface** (both yield `FrameItem`):
- `"video"` → real incident / post-flight drone footage, sampled every N seconds
- `"images"` → folder of labelled dataset frames (HERIDAL, SARD, WiSARD, …)

**Memory design:** frames are streamed one at a time; only candidate/audit frames are written as JPEG.

Also includes ground-truth label readers (YOLO `.txt`, Pascal-VOC `.xml`, RetinaNet CSV) used by Cell 2A preview and Cell 6 evaluation.


In [ ]:
# ==============================================================================
# AAROHAN | CELL 2 — Frame Extraction & Sampling Engine
# ==============================================================================
# Two input modes, one interface (both yield FrameItem objects):
#   • "video"  → real incident / post-flight drone footage, sampled every N seconds
#   • "images" → a folder of labelled dataset frames (HERIDAL, SARD, WiSARD, ...)
#
# Memory design (Apple unified memory): frames are STREAMED one at a time and never
# accumulated in RAM. Only candidate / audit frames are written to disk as JPEG.
# Edge cases handled: missing path, empty file, unreadable/corrupt container,
# fps <= 0 or NaN, bogus frame counts, mid-stream decode failures, truncated files.
# ------------------------------------------------------------------------------
from typing import NamedTuple

SUPPORTED_VIDEO_EXTS = {".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm", ".mts", ".m2ts"}
SUPPORTED_IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


class VideoError(RuntimeError):
    """Raised for any unrecoverable problem with the input video / image folder."""


class FrameItem(NamedTuple):
    index: int              # frame number (video) or position in sorted folder (images)
    timestamp_s: float      # seconds into the video (images: position index, no real clock)
    frame: np.ndarray       # BGR uint8 image
    source_name: str        # video stem, or unique image id
    source_path: str        # original file path (used to look up ground truth)


@dataclass
class SamplerStats:
    """Counters filled while streaming — saved in run_summary.json."""
    frames_grabbed: int = 0
    frames_emitted: int = 0
    corrupt_frames_skipped: int = 0
    grab_failures: int = 0
    truncated: bool = False


@dataclass(frozen=True)
class VideoMetadata:
    path: str
    width: int
    height: int
    fps: float
    fps_source: str            # "container" or "fallback"
    frame_count: int
    frame_count_source: str    # "container" or "counted"
    duration_s: float
    fourcc: str
    file_size_mb: float

    def as_dict(self) -> dict:
        return asdict(self)


def format_hms(seconds: float) -> str:
    """12.5 -> '00:00:12.500'"""
    seconds = max(0.0, float(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{int(h):02d}:{int(m):02d}:{s:06.3f}"


# ----------------------------------------------------------------------------
# Disk helpers (unicode-safe: avoid cv2.imread/imwrite on non-ASCII macOS paths)
# ----------------------------------------------------------------------------
def save_frame_jpeg(frame: np.ndarray, path: Path, quality: int = 92) -> None:
    ok, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, int(quality)])
    if not ok:
        raise VideoError(f"JPEG encoding failed for {path.name}")
    Path(path).write_bytes(buf.tobytes())


def load_frame(path) -> Optional[np.ndarray]:
    """Load an image as BGR uint8. Returns None if the file is missing/corrupt."""
    try:
        data = np.frombuffer(Path(path).read_bytes(), dtype=np.uint8)
        img = cv2.imdecode(data, cv2.IMREAD_COLOR)
        return img if img is not None and img.size > 0 else None
    except Exception:
        return None


class LRUCache:
    """Tiny bounded cache so the review UI never grows RAM unboundedly."""

    def __init__(self, max_items: int = 8):
        self.max_items = max_items
        self._d: "OrderedDict[str, object]" = OrderedDict()

    def get(self, key):
        if key in self._d:
            self._d.move_to_end(key)
            return self._d[key]
        return None

    def put(self, key, value) -> None:
        self._d[key] = value
        self._d.move_to_end(key)
        while len(self._d) > self.max_items:
            self._d.popitem(last=False)

    def clear(self) -> None:
        self._d.clear()


# ----------------------------------------------------------------------------
# Video probing
# ----------------------------------------------------------------------------
def validate_video_path(path) -> Path:
    p = Path(path).expanduser()
    if not p.exists():
        raise FileNotFoundError(f"Video not found: {p}")
    if not p.is_file():
        raise VideoError(f"Not a file: {p}")
    if p.stat().st_size == 0:
        raise VideoError(f"Video file is empty (0 bytes): {p.name}")
    if p.suffix.lower() not in SUPPORTED_VIDEO_EXTS:
        log.warning("Unusual video extension '%s' — will still try to open it.", p.suffix)
    return p.resolve()


def _fourcc_to_str(value: float) -> str:
    try:
        return int(value).to_bytes(4, "little").decode("ascii", errors="ignore").strip() or "n/a"
    except Exception:
        return "n/a"


def _count_frames(path: Path) -> int:
    """Slow-but-exact frame count (only used when the container metadata is useless)."""
    cap = cv2.VideoCapture(str(path))
    n = 0
    try:
        while cap.grab():
            n += 1
    finally:
        cap.release()
    return n


def probe_video(path, fallback_fps: float = 30.0) -> VideoMetadata:
    """Validate the file, decode one frame, and return reliable metadata."""
    p = validate_video_path(path)
    cap = cv2.VideoCapture(str(p))
    try:
        if not cap.isOpened():
            raise VideoError(f"OpenCV cannot open '{p.name}' (unsupported codec or corrupted container).")
        ok, first = cap.read()
        if not ok or first is None or first.size == 0:
            raise VideoError(f"'{p.name}' opened but its first frame cannot be decoded (corrupted file).")
        height, width = first.shape[:2]

        fps = cap.get(cv2.CAP_PROP_FPS)
        fps_source = "container"
        if not (isinstance(fps, (int, float)) and math.isfinite(fps)) or fps < 0.01 or fps > 1000:
            log.warning("Container reports invalid fps (%s) — using fallback %.1f fps.", fps, fallback_fps)
            fps, fps_source = float(fallback_fps), "fallback"

        raw_n = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        n_frames = int(raw_n) if isinstance(raw_n, (int, float)) and math.isfinite(raw_n) else 0
        fourcc = _fourcc_to_str(cap.get(cv2.CAP_PROP_FOURCC))
    finally:
        cap.release()

    count_source = "container"
    if n_frames <= 0 or n_frames > 10_000_000:
        log.warning("Frame count missing/implausible — counting frames (one-off, may take a moment).")
        n_frames, count_source = _count_frames(p), "counted"
    if n_frames <= 0:
        raise VideoError(f"'{p.name}' contains no decodable frames.")

    return VideoMetadata(
        path=str(p), width=int(width), height=int(height), fps=float(fps), fps_source=fps_source,
        frame_count=int(n_frames), frame_count_source=count_source,
        duration_s=float(n_frames / fps), fourcc=fourcc,
        file_size_mb=round(p.stat().st_size / 1024 ** 2, 2),
    )


def plan_sampling(meta: VideoMetadata, interval_s: float, start_s: float = 0.0,
                  end_s: Optional[float] = None, max_frames: Optional[int] = None):
    """Translate seconds into frame indices. Returns (start_idx, end_idx, step, expected_samples)."""
    if not (isinstance(interval_s, (int, float)) and interval_s > 0):
        raise ValueError("sample_interval_s must be > 0")
    step = max(1, int(round(interval_s * meta.fps)))
    start_idx = max(0, int(round(start_s * meta.fps)))
    end_idx = meta.frame_count if end_s is None else min(meta.frame_count, int(round(end_s * meta.fps)))
    if start_idx >= end_idx:
        raise ValueError(f"Empty sampling window: start={start_idx}, end={end_idx} (frames).")
    expected = int(math.ceil((end_idx - start_idx) / step))
    if max_frames:
        expected = min(expected, int(max_frames))
    return start_idx, end_idx, step, expected


# ----------------------------------------------------------------------------
# Streaming samplers
# ----------------------------------------------------------------------------
def iter_sampled_frames(path, interval_s: float = 1.0, start_s: float = 0.0,
                        end_s: Optional[float] = None, max_frames: Optional[int] = None,
                        fallback_fps: float = 30.0, max_consecutive_failures: int = 30,
                        meta: Optional[VideoMetadata] = None,
                        stats: Optional[SamplerStats] = None) -> Iterator[FrameItem]:
    """Yield one frame every `interval_s` seconds without ever holding more than one frame.

    Uses sequential cap.grab() (cheap) and cap.retrieve() only on sampled frames —
    more reliable than random seeking on long-GOP drone footage (H.264/H.265).
    """
    meta = meta or probe_video(path, fallback_fps)
    stats = stats if stats is not None else SamplerStats()
    start_idx, end_idx, step, _ = plan_sampling(meta, interval_s, start_s, end_s, max_frames)
    stem = Path(meta.path).stem

    cap = cv2.VideoCapture(meta.path)
    try:
        if not cap.isOpened():
            raise VideoError(f"OpenCV cannot re-open '{stem}'.")
        idx = 0
        if start_idx > 0:                         # fast (approximate) seek, then verify position
            cap.set(cv2.CAP_PROP_POS_FRAMES, start_idx)
            idx = int(cap.get(cv2.CAP_PROP_POS_FRAMES) or 0)
        next_sample = max(idx, start_idx)
        failures = 0
        emitted = 0
        while idx < end_idx:
            if not cap.grab():                    # EOF or undecodable packet
                failures += 1
                stats.grab_failures += 1
                if failures >= max_consecutive_failures:
                    stats.truncated = idx < end_idx * 0.98
                    if stats.truncated:
                        log.warning("Stream ended early at frame %d/%d — file may be truncated or corrupt.",
                                    idx, end_idx)
                    break
                idx += 1
                continue
            failures = 0
            stats.frames_grabbed += 1
            if idx == next_sample:
                ok, frame = cap.retrieve()
                if ok and frame is not None and frame.size > 0:
                    stats.frames_emitted += 1
                    emitted += 1
                    yield FrameItem(idx, idx / meta.fps, frame, stem, meta.path)
                    if max_frames and emitted >= max_frames:
                        break
                else:
                    stats.corrupt_frames_skipped += 1
                    log.warning("Skipped undecodable frame #%d.", idx)
                next_sample += step
            idx += 1
    finally:
        cap.release()                             # always free the decoder / its buffers
        gc.collect()


def natural_key(p: Path):
    """Sort 'img2' before 'img10'."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", str(p))]


def list_images(folder) -> List[Path]:
    folder = Path(folder).expanduser()
    if not folder.exists():
        raise FileNotFoundError(f"Image folder not found: {folder}")
    if not folder.is_dir():
        raise VideoError(f"Not a directory: {folder}")
    files = [p for p in folder.rglob("*") if p.suffix.lower() in SUPPORTED_IMAGE_EXTS
             and not p.name.startswith(".") and "__MACOSX" not in p.parts]
    files.sort(key=natural_key)
    if not files:
        raise VideoError(f"No images ({sorted(SUPPORTED_IMAGE_EXTS)}) found under {folder}")
    return files


def image_id(path: Path, root: Path) -> str:
    """Stable, filesystem-safe unique id for an image inside the dataset folder."""
    rel = path.relative_to(root).with_suffix("")
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(rel)).strip("_")


def iter_image_folder(folder, max_frames: Optional[int] = None, every_nth: int = 1,
                      stats: Optional[SamplerStats] = None) -> Iterator[FrameItem]:
    """Dataset mode: yield each image like a sampled video frame (timestamp = position)."""
    folder = Path(folder).expanduser().resolve()
    stats = stats if stats is not None else SamplerStats()
    files = list_images(folder)[::max(1, int(every_nth))]
    if max_frames:
        files = files[:int(max_frames)]
    for pos, path in enumerate(files):
        stats.frames_grabbed += 1
        img = load_frame(path)
        if img is None:
            stats.corrupt_frames_skipped += 1
            log.warning("Skipped unreadable image: %s", path.name)
            continue
        stats.frames_emitted += 1
        yield FrameItem(pos, float(pos), img, image_id(path, folder), str(path))


def describe_source(cfg) -> Tuple[Optional[VideoMetadata], int]:
    """Print what Cell 3 is about to process. Returns (metadata or None, expected frames)."""
    if cfg.source_mode == "video":
        meta = probe_video(cfg.video_path, cfg.fallback_fps)
        _, _, step, expected = plan_sampling(meta, cfg.sample_interval_s, cfg.start_s, cfg.end_s, cfg.max_frames)
        print(f"  {Path(meta.path).name}: {meta.width}x{meta.height} @ {meta.fps:.2f} fps ({meta.fps_source}), "
              f"{meta.frame_count} frames ({meta.frame_count_source}), {format_hms(meta.duration_s)}, "
              f"{meta.file_size_mb} MB, codec={meta.fourcc}")
        print(f"   Sampling every {cfg.sample_interval_s}s (= every {step} frames) → ~{expected} frames to analyse.")
        return meta, expected
    if cfg.source_mode == "images":
        files = list_images(cfg.images_dir)
        n = min(len(files), cfg.max_frames) if cfg.max_frames else len(files)
        print(f"  Image-folder mode: {len(files)} images under {cfg.images_dir} → analysing {n}.")
        return None, n
    raise ValueError("CFG.source_mode must be 'video' or 'images'")


def iter_source(cfg, meta: Optional[VideoMetadata], stats: SamplerStats) -> Iterator[FrameItem]:
    if cfg.source_mode == "video":
        return iter_sampled_frames(cfg.video_path, cfg.sample_interval_s, cfg.start_s, cfg.end_s,
                                   cfg.max_frames, cfg.fallback_fps, meta=meta, stats=stats)
    return iter_image_folder(cfg.images_dir, cfg.max_frames, stats=stats)


# ----------------------------------------------------------------------------
# Ground-truth label readers (used by Cell 2B preview and Cell 6 evaluation)
# Supports YOLO .txt (normalised cx cy w h) and Pascal-VOC .xml. Every class is treated as "person".
# ----------------------------------------------------------------------------
import xml.etree.ElementTree as ET

def _read_yolo_labels(path: Path, img_w: int, img_h: int) -> np.ndarray:
    boxes = []
    for line in path.read_text().splitlines():
        parts = line.split()
        if len(parts) >= 5:
            try:
                cx, cy, w, h = (float(v) for v in parts[1:5])
            except ValueError:
                continue
            boxes.append([(cx - w / 2) * img_w, (cy - h / 2) * img_h, (cx + w / 2) * img_w, (cy + h / 2) * img_h])
    return np.array(boxes, dtype=np.float32).reshape(-1, 4)


def _read_voc_labels(path: Path) -> np.ndarray:
    boxes = []
    for obj in ET.parse(path).getroot().iter("object"):
        bb = obj.find("bndbox")
        if bb is None:
            continue
        try:
            boxes.append([float(bb.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax")])
        except (TypeError, ValueError):
            continue
    return np.array(boxes, dtype=np.float32).reshape(-1, 4)


def _read_retinanet_csv(path: Path) -> Dict[str, np.ndarray]:
    """keras-retinanet CSV rows: image_path,x1,y1,x2,y2,class  (empty box fields = image without objects)."""
    out: Dict[str, List[List[float]]] = {}
    with open(path, newline="", encoding="utf-8", errors="ignore") as fh:
        for row in csv.reader(fh):
            if len(row) < 5 or not row[0].strip():
                continue
            stem = Path(row[0].strip()).stem
            out.setdefault(stem, [])
            try:
                out[stem].append([float(v) for v in row[1:5]])
            except ValueError:
                continue                          # header or blank-box row -> image has zero objects
    return {k: np.array(v, dtype=np.float32).reshape(-1, 4) for k, v in out.items()}


def build_label_index(labels_dir) -> Dict[str, object]:
    """Map image stem -> label source: a Path (.xml preferred, else .txt) or a ready (N,4) array (.csv).
    Every ground-truth class is treated as 'person'."""
    index: Dict[str, object] = {}
    for p in sorted(Path(labels_dir).expanduser().rglob("*")):
        if p.name.startswith(".") or p.stem.lower() in ("classes", "readme"):
            continue
        ext = p.suffix.lower()
        if ext == ".csv":
            try:
                for stem, boxes in _read_retinanet_csv(p).items():
                    index.setdefault(stem, boxes)
            except Exception as exc:
                log.warning("Unreadable CSV labels %s (%s)", p.name, exc)
        elif ext in (".xml", ".txt"):
            if p.stem not in index or ext == ".xml":
                index[p.stem] = p
    return index


def load_ground_truth(all_frames: List[dict], labels_dir) -> Dict[str, np.ndarray]:
    """frame_id -> (N, 4) GT boxes, only for frames that have a label entry."""
    index = build_label_index(labels_dir)
    gt: Dict[str, np.ndarray] = {}
    for fr in all_frames:
        stem = Path(fr["source_path"]).stem
        lp = index.get(stem)
        if lp is None:
            continue
        try:
            if isinstance(lp, np.ndarray):
                gt[fr["frame_id"]] = lp
            elif lp.suffix.lower() == ".xml":
                gt[fr["frame_id"]] = _read_voc_labels(lp)
            else:
                with Image.open(fr["source_path"]) as im:        # header read only — cheap
                    w, h = im.size
                gt[fr["frame_id"]] = _read_yolo_labels(lp, w, h)
        except Exception as exc:
            log.warning("Unreadable label for %s (%s)", stem, exc)
    return gt


# ----------------------------------------------------------------------------
# Self-check: describe the configured source if it exists (never crashes the cell)
# ----------------------------------------------------------------------------
try:
    describe_source(CFG)
except (FileNotFoundError, VideoError, ValueError) as exc:
    print(f"ℹ Input not ready yet: {exc}")
    print("  → Put your footage at CFG.video_path (source_mode='video') or a dataset folder at "
          "CFG.images_dir (source_mode='images'), then re-run this cell.")


---
## Cell 2A — Choose your data (real aerial SAR datasets / real incident footage)

Edit the **last lines** of the next code cell:

| Track | Call | Ground truth? |
|-------|------|---------------|
| **A — Validation** | `use_dataset(...)` | Yes — Cell 6 reports objective P/R |
| **B — Field test** | `use_real_incident_video(...)` | No — human reviewers + audit produce the truth |

Public SAR datasets use volunteers/actors on real terrain (real casualty imagery is not published). Use Track B for authorised genuine incident footage.


In [ ]:
# ==============================================================================
# AAROHAN | CELL 2B — Data Source: Real Aerial SAR Datasets + Real Incident Footage
# ==============================================================================
# TRACK A — VALIDATION with ground truth (real drone imagery, labelled persons):
#     HERIDAL, SARD, WiSARD  → CFG.source_mode = "images", CFG.labels_dir = <labels>
#     Cell 6 then reports objective precision / recall against the human-drawn labels.
# TRACK B — FIELD TEST on real incident footage (no labels exist):
#     CFG.source_mode = "video", CFG.video_path = <your footage>
#     The human reviewers + audit sample (Cells 4–6) PRODUCE the ground truth.
#
# Honest note on "real": the public annotated SAR datasets were captured by drones over
# real terrain with real people, but the people are volunteers / actors simulating
# casualties (real casualties are not published for privacy reasons). Use Track B for
# genuine incident footage that you are authorised to use.
#
# These datasets cannot be fetched with one anonymous URL (login or Google-Drive pages),
# so this cell gives you an exact download recipe and then automates everything after the
# download: extraction, image/label discovery, CFG wiring and a visual label check.
# ------------------------------------------------------------------------------
import tarfile
import zipfile

DATASET_CATALOG = {
    "HERIDAL": {
        "what": "Land-SAR aerial images for person detection (wilderness / rural terrain, mostly simulated rescue scenarios, "
                "very small persons) — the most SAR-realistic AND freely downloadable benchmark. License CC BY 3.0.",
        "get": "download_zenodo_record(5662351, max_size_gb=None)  = 'HERIDAL in keras-retinanet PASCAL VOC format' (8.3 GB, 3,338 files, Pascal-VOC XML labels)  "
               "| original page: http://ipsar.fesb.unist.hr/HERIDAL%20database.html",
        "cite": "Bozic-Stulic, Marusic, Gotovac, 'Deep learning approach in aerial imagery for supporting land SAR missions', IJCV 127(9), 2019",
    },
    "SARD": {
        "what": "1,981 labelled frames from ~35 min of drone video; actors simulate exhausted / injured persons and standing, walking, "
                "running, sitting, lying poses; 'Corr' extension adds fog, snow, ice and motion blur. Boxes drawn with LabelImg.",
        "get": "https://ieee-dataport.org/documents/search-and-rescue-image-dataset-person-detection-sard  "
               "(IEEE DataPort login + subscription required; file SARD-fixed.zip, 4.43 GB) — manual download",
        "cite": "Sambolek & Ivasic-Kos, SARD, IEEE DataPort, 2021, doi:10.21227/ahxm-k331",
    },
    "WiSARD": {
        "what": "~56,000 labelled visual + thermal UAV images across terrains, seasons, weather and lighting (wilderness SAR). "
                "Start with the ~1 GB multi-modal sample before the ~40 GB full set.",
        "get": "https://sites.google.com/uw.edu/wisard/  (download links on that page) — manual download",
        "cite": "Broyles, Hayner, Leung, 'WiSARD', IROS 2022, doi:10.1109/IROS47612.2022.9981298",
    },
    "NOMAD": {
        "what": "Occlusion benchmark: 100 actors hiding / lying / walking at several altitudes, 5.4k video, 42,825 labelled frames "
                "with visibility levels. Best for testing 'person partly hidden' behaviour.",
        "get": "https://github.com/ArtRuss/NOMAD",
        "cite": "Russell Bernal, Scheirer, Cleland-Huang, NOMAD, arXiv:2309.09518",
    },
    "AIR-demo-images": {
        "what": "SMALL (25 MB) sample of HERIDAL test images released with the AIR project — best first smoke test of the whole pipeline. CC BY 3.0. "
                "(images only; use the full HERIDAL set for labels)",
        "get": "download_zenodo_record(5662638)  → test.zip  → use_dataset('HERIDAL-demo', <that zip>)",
        "cite": "Pyrro, 'AIR demo images', Zenodo, doi:10.5281/zenodo.5662637",
    },
    "AIR-demo-videos": {
        "what": "Aerial SAR demo videos (Finnish voluntary rescue service material) released with the AIR project. CC BY 4.0. "
                "WARNING: ONE 8.3 GB videos.zip — needs ~17 GB free disk; not a quick test.",
        "get": "download_zenodo_record(5662512, max_size_gb=None) → extract_archive(...) → find_videos(...)",
        "cite": "Pyrro, Naseri, Jung, 'AIR: Aerial Inspection RetinaNet for Land Search and Rescue Missions', Aalto Univ. 2021",
    },
}


def download_zenodo_record(record_id: int, dest_dir=None, only_ext: Optional[Tuple[str, ...]] = None,
                           max_size_gb: Optional[float] = 2.0) -> List[Path]:
    """Download the files of a public Zenodo record (open access, no login).

    only_ext    : keep only these extensions, e.g. (".zip",)  — None keeps everything
    max_size_gb : safety guard — files bigger than this are LISTED but skipped. Pass None to allow huge files.
    Always prints what the record contains, so an empty result is explained instead of silent.
    """
    import urllib.request
    dest_dir = Path(dest_dir or (CFG.project_dir / "downloads" / f"zenodo_{record_id}")).expanduser()
    dest_dir.mkdir(parents=True, exist_ok=True)
    with urllib.request.urlopen(f"https://zenodo.org/api/records/{record_id}", timeout=60) as resp:
        meta = json.loads(resp.read().decode("utf-8"))
    files = meta.get("files", [])
    if isinstance(files, dict):                                  # tolerate alternative API shapes
        files = files.get("entries", [])
    if not files:
        raise VideoError(f"Zenodo record {record_id} lists no downloadable files (restricted or removed).")
    print(f"Zenodo record {record_id}: {len(files)} file(s)")
    for f in files:
        print(f"   • {f.get('key') or f.get('filename')}   {int(f.get('size') or 0) / 1024 ** 3:.2f} GB")
    saved: List[Path] = []
    for f in files:
        name = f.get("key") or f.get("filename")
        url = (f.get("links") or {}).get("self") or (f.get("links") or {}).get("download")
        size = int(f.get("size") or 0)
        if not name or not url:
            continue
        if only_ext and Path(name).suffix.lower() not in only_ext:
            print(f"   skipped (extension filter {only_ext}): {name}")
            continue
        if max_size_gb is not None and size > max_size_gb * 1024 ** 3:
            print(f"   skipped (> {max_size_gb} GB safety limit): {name} — pass max_size_gb=None if you really want it "
                  f"(and make sure you have ~2x its size free on disk).")
            continue
        target = dest_dir / name
        if target.exists() and (size == 0 or target.stat().st_size == size):
            print(f"✔ already downloaded: {name}")
            saved.append(target)
            continue
        print(f"⬇ {name}  ({size / 1024 ** 2:.1f} MB)")
        tmp = target.with_suffix(target.suffix + ".part")
        with urllib.request.urlopen(url, timeout=120) as r, open(tmp, "wb") as out, \
                tqdm(total=size or None, unit="B", unit_scale=True, desc=name[:30]) as bar:
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                out.write(chunk)
                bar.update(len(chunk))
        os.replace(tmp, target)
        saved.append(target)
    if not saved:
        print("⚠ Nothing was downloaded — see the 'skipped' lines above (extension filter or size limit).")
    return saved


def find_videos(root) -> List[Path]:
    """All video files under a folder (use after extract_archive on a downloaded videos.zip)."""
    root = Path(root).expanduser()
    return sorted(p for p in root.rglob("*") if p.suffix.lower() in SUPPORTED_VIDEO_EXTS
                  and not p.name.startswith(".") and "__MACOSX" not in p.parts)


def show_dataset_catalog() -> None:
    for name, d in DATASET_CATALOG.items():
        print(f"■ {name}\n   {d['what']}\n   get : {d['get']}\n   cite: {d['cite']}\n")


def _safe_extract_zip(zf: zipfile.ZipFile, dest: Path) -> None:
    dest = dest.resolve()
    for member in zf.namelist():
        target = (dest / member).resolve()
        if not str(target).startswith(str(dest)):          # zip-slip guard
            raise VideoError(f"Unsafe path inside archive: {member}")
    zf.extractall(dest)


def extract_archive(archive, dest) -> Path:
    """Extract .zip / .tar(.gz) into dest (skips work if dest already has files)."""
    archive, dest = Path(archive).expanduser(), Path(dest).expanduser()
    if not archive.exists():
        raise FileNotFoundError(f"Archive not found: {archive}")
    dest.mkdir(parents=True, exist_ok=True)
    if any(dest.iterdir()):
        print(f"ℹ {dest} is not empty — assuming already extracted.")
        return dest
    print(f"Extracting {archive.name} → {dest} (this can take a few minutes for multi-GB archives)...")
    if zipfile.is_zipfile(archive):
        with zipfile.ZipFile(archive) as zf:
            _safe_extract_zip(zf, dest)
    elif tarfile.is_tarfile(archive):
        with tarfile.open(archive) as tf:
            tf.extractall(dest, filter="data") if hasattr(tarfile, "data_filter") else tf.extractall(dest)
    else:
        raise VideoError(f"Unsupported archive type: {archive.name}")
    return dest


def discover_dataset_dirs(root) -> dict:
    """Find the folder holding most images and the folder holding most label files (.xml / .txt)."""
    root = Path(root).expanduser()
    img_dirs: Dict[Path, int] = {}
    lbl_dirs: Dict[Path, int] = {}
    for p in root.rglob("*"):
        if not p.is_file() or p.name.startswith(".") or "__MACOSX" in p.parts:
            continue
        ext = p.suffix.lower()
        if ext in SUPPORTED_IMAGE_EXTS:
            img_dirs[p.parent] = img_dirs.get(p.parent, 0) + 1
        elif ext in (".xml", ".txt", ".csv") and p.stem.lower() not in ("classes", "readme"):
            lbl_dirs[p.parent] = lbl_dirs.get(p.parent, 0) + 1
    if not img_dirs:
        raise VideoError(f"No images found under {root}")
    # Use the common parent so datasets with several image sub-folders (train/val/Corr/...) are all included.
    best_img = max(img_dirs, key=img_dirs.get)
    images_root = Path(os.path.commonpath([str(d) for d, n in img_dirs.items() if n >= 0.1 * img_dirs[best_img]]))
    labels_root = None
    if lbl_dirs:
        best_lbl = max(lbl_dirs, key=lbl_dirs.get)
        labels_root = Path(os.path.commonpath([str(d) for d, n in lbl_dirs.items() if n >= 0.1 * lbl_dirs[best_lbl]]))
    return {"images_dir": images_root, "labels_dir": labels_root,
            "n_images": sum(img_dirs.values()), "n_label_files": sum(lbl_dirs.values())}


def preview_dataset(images_dir, labels_dir, n: int = 6, seed: int = 0) -> None:
    """Draw ground-truth boxes on random images — verify the labels line up before trusting any metric."""
    files = list_images(images_dir)
    rnd = random.Random(seed)
    pick = rnd.sample(files, min(n, len(files)))
    frames = [{"frame_id": image_id(p, Path(images_dir).resolve()), "source_path": str(p)} for p in pick]
    gt = load_ground_truth(frames, labels_dir) if labels_dir else {}
    cols = 3
    rows = int(math.ceil(len(pick) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5.2 * cols, 3.8 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, fr, p in zip(axes, frames, pick):
        img = load_frame(p)
        if img is None:
            continue
        boxes = gt.get(fr["frame_id"])
        dets = np.hstack([boxes, np.ones((len(boxes), 1), np.float32)]) if boxes is not None and len(boxes) else np.zeros((0, 5))
        ax.imshow(annotate_frame(img, dets, max_side=900, draw_labels=False))
        ax.set_title(f"{p.name} — {0 if boxes is None else len(boxes)} labelled person(s)"
                     + ("" if boxes is not None else "  [NO LABEL FILE]"), fontsize=9)
    plt.tight_layout()
    plt.show()


def use_dataset(name: str, path, label_note: str = "", every_nth: int = 1, max_frames: Optional[int] = None,
                preview: bool = True) -> dict:
    """One call to wire a downloaded dataset into CFG.

    path : the downloaded archive (.zip/.tar) OR the already-extracted folder.
    Afterwards re-run Cell 3 (and everything after it).
    """
    path = Path(path).expanduser()
    root = path
    if path.is_file():
        root = extract_archive(path, CFG.project_dir / "datasets" / name)
    found = discover_dataset_dirs(root)
    CFG.source_mode = "images"
    CFG.images_dir = found["images_dir"]
    CFG.labels_dir = found["labels_dir"]
    if CFG.labels_dir is not None and not list(Path(CFG.labels_dir).glob("*")):
        CFG.labels_dir = None
    CFG.max_frames = max_frames
    CFG.data_source_label = f"{name} (real aerial SAR dataset{', ' + label_note if label_note else ''})"
    print(f"✔ {name}: {found['n_images']} images, {found['n_label_files']} label files")
    print(f"  images_dir = {CFG.images_dir}\n  labels_dir = {CFG.labels_dir}")
    if CFG.labels_dir is None:
        print("  ⚠ No label files found — Track-A ground-truth metrics will be skipped (reviewer metrics still work).")
    if preview and CFG.labels_dir is not None:
        preview_dataset(CFG.images_dir, CFG.labels_dir)
    if name.upper() == "HERIDAL":
        print("  Tip: HERIDAL persons are tiny — keep CFG.use_tiling=True and try CFG.tile_imgsz=960 / CFG.conf=0.15.")
    return found


def use_real_incident_video(video_path, incident_label: str, interval_s: float = 1.0) -> None:
    """Track B: point the pipeline at real incident footage (no ground truth; humans create it)."""
    CFG.source_mode = "video"
    CFG.video_path = Path(video_path).expanduser()
    CFG.labels_dir = None
    CFG.sample_interval_s = interval_s
    CFG.data_source_label = f"real incident footage: {incident_label}"
    describe_source(CFG)


# ----------------------------------------------------------------------------
# CHOOSE YOUR DATA — uncomment ONE line, then run Cell 3.
# ----------------------------------------------------------------------------
show_dataset_catalog()

# FIRST SMOKE TEST (25 MB, 1 minute): sample HERIDAL images from the AIR project.
# files = download_zenodo_record(5662638)               # → test.zip
# use_dataset("HERIDAL-demo", files[0], label_note="25 MB sample, images only", preview=False)

# TRACK A — full HERIDAL with labels (8.3 GB download, ~17 GB free disk needed; Pascal-VOC XML labels):
# files = download_zenodo_record(5662351, max_size_gb=None)
# use_dataset("HERIDAL", files[0], label_note="land-SAR benchmark, CC BY 3.0", max_frames=200)

# TRACK A (manual download first): SARD / WiSARD
# use_dataset("SARD",   Path.home() / "Downloads" / "SARD-fixed.zip",     label_note="actors simulating casualties")
# use_dataset("WiSARD", Path.home() / "Downloads" / "WiSARD_sample.zip",  label_note="wilderness SAR, visual subset")

# TRACK B — real SAR video: your own authorised footage (or the 8.3 GB AIR demo videos.zip):
# use_real_incident_video(Path.home() / "AAROHAN" / "videos" / "incident_flight.mp4", "<incident name, date, source of footage>")
# z = download_zenodo_record(5662512, max_size_gb=None); root = extract_archive(z[0], CFG.project_dir / "datasets" / "AIR_videos")
# vids = find_videos(root); print(vids); use_real_incident_video(vids[0], "AIR demo video (Finnish voluntary rescue)")


---
## Cell 2B — HERIDAL test split only (optional ground-truth benchmark)

Opt-in: set `RUN_HERIDAL_TEST_DOWNLOAD = True`, run once.

- 101 labelled images, ~0.5 GB
- Fetched selectively from the remote zip (no full 8 GB download)
- Wires `CFG` to image mode + labels for Cell 3 → Cell 6 GT metrics


In [ ]:
# ==============================================================================
# AAROHAN | CELL 2C — HERIDAL TEST SPLIT (101 labelled images, ~0.5 GB instead of the 8.3 GB zip)
# ==============================================================================
# Reads ONLY the zip's index plus the test images + their Pascal-VOC XML labels over HTTP
# range requests (package: remotezip). Resumable: finished files are skipped on re-run.
# Opt-in: set RUN_HERIDAL_TEST_DOWNLOAD = True below, run the cell, then run Cell 3 and Cell 6
# (Cells 4-5 are optional — Cell 6 benchmarks directly against the labels).
# Dataset: HERIDAL (Bozic-Stulic et al., IJCV 2019), CC BY 3.0, Zenodo doi:10.5281/zenodo.5662351
# ------------------------------------------------------------------------------
HERIDAL_URL = "https://zenodo.org/records/5662351/files/heridal_keras_retinanet_voc.zip?download=1"
HERIDAL_BASE = "heridal_keras_retinanet_voc/"
RUN_HERIDAL_TEST_DOWNLOAD = True         # <- set True to download (about 0.5 GB)


def fetch_subset(zf, ids, img_dir, lbl_dir, base):
    """Copy only the given image ids (+ their XML labels) out of an open (remote) zip.
    Resumable: files whose size already matches are skipped. Returns (files_ready, missing_ids)."""
    info = {i.filename: i for i in zf.infolist()}
    by_stem = {Path(n).stem: n for n in info if n.startswith(base + "JPEGImages/") and not n.endswith("/")}
    img_dir, lbl_dir = Path(img_dir), Path(lbl_dir)
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    jobs, missing = [], []
    for i in ids:
        xml = base + "Annotations/" + i + ".xml"
        if i not in by_stem or xml not in info:
            missing.append(i)
            continue
        jobs.append((by_stem[i], img_dir / Path(by_stem[i]).name))
        jobs.append((xml, lbl_dir / (i + ".xml")))
    done = 0
    for name, target in tqdm(jobs, desc="HERIDAL test subset", unit="file"):
        if target.exists() and target.stat().st_size == info[name].file_size:
            done += 1
            continue
        data = zf.read(name)
        tmp = target.with_suffix(target.suffix + ".part")
        tmp.write_bytes(data)
        os.replace(tmp, target)
        done += 1
    return done, missing


def prepare_heridal_test_split(opener=None, conf: float = 0.10) -> bool:
    """Download the test split and point CFG at it. `opener` lets tests inject a local zip."""
    if opener is None:
        from remotezip import RemoteZip
        opener = lambda: RemoteZip(HERIDAL_URL)          # noqa: E731
    img_dir = CFG.project_dir / "datasets" / "HERIDAL_test" / "images"
    lbl_dir = CFG.project_dir / "datasets" / "HERIDAL_test" / "labels"
    with opener() as z:
        ids = z.read(HERIDAL_BASE + "ImageSets/Main/test.txt").decode().split()
        done, missing = fetch_subset(z, ids, img_dir, lbl_dir, HERIDAL_BASE)
    n_img, n_lbl = len(list(img_dir.glob("*"))), len(list(lbl_dir.glob("*.xml")))
    print(f"files ready: {done} (expected {2 * len(ids)}) | missing: {missing} | images: {n_img} | labels: {n_lbl}")
    if missing or n_img != len(ids) or n_lbl != len(ids):
        print("⚠ Incomplete — run this cell again (finished files are skipped) before running Cell 3.")
        return False
    CFG.source_mode = "images"
    CFG.images_dir, CFG.labels_dir = img_dir, lbl_dir
    CFG.max_frames = None
    CFG.data_source_label = f"HERIDAL test split, {len(ids)} images (CC BY 3.0, Zenodo 5662351)"
    CFG.run_id = datetime.now().strftime("%Y%m%d_%H%M%S")          # new run: never mix with older decisions
    CFG.run_dir.mkdir(parents=True, exist_ok=True)
    CFG.frames_dir.mkdir(parents=True, exist_ok=True)
    CFG.conf = conf
    CFG.audit_fraction = 0.0                                       # ground truth exists → no audit sampling needed
    print(f"✔ CFG ready: {CFG.source_mode} | {CFG.images_dir.parent.name} | conf={CFG.conf} | run_id={CFG.run_id}")
    print("  Next: run Cell 3, then Cell 6 (Cells 4–5 optional).")
    return True


if RUN_HERIDAL_TEST_DOWNLOAD:
    prepare_heridal_test_split()
else:
    print("ℹ Cell 2C is idle. Set RUN_HERIDAL_TEST_DOWNLOAD = True and re-run to download the HERIDAL test split.")


---
## Cell 3 — MPS YOLO inference & candidate selection

**Tiled inference is on by default** — aerial persons are often only a few pixels wide.

Flow:
1. Load YOLO weights (auto-download if needed)
2. Full-frame pass + optional overlapping tiles → merge NMS
3. Size filters (`min_box_px`, `max_box_frac`)
4. Frames with ≥ `min_detections` → **candidates**
5. Random share of empty frames → **audit** sample (for recall estimate)
6. Write candidate JPEGs + `candidates.json` + `all_frames.json` + `run_summary.json`


In [ ]:
# ==============================================================================
# AAROHAN | CELL 3 — MPS-Accelerated YOLO Inference & Candidate Frame Selection
# ==============================================================================
# Pipeline per sampled frame:
#   full-frame pass  (+ optional tiled passes for tiny aerial persons)
#   → class filter (COCO 0 = person) → conf / NMS → cross-tile merge → box sanity filters
#   → candidate (>= min_detections)  |  audit sample of detection-free frames
#
# Why audit frames? A reviewer only sees frames the model flagged, so a missed person
# is invisible. Randomly sampling some "empty" frames lets Cell 6 ESTIMATE recall and
# false-positive rate honestly instead of pretending they can be read off the review log.
# ------------------------------------------------------------------------------

@dataclass
class FrameRecord:
    """One frame that enters the human review queue."""
    frame_id: str
    frame_index: int
    timestamp_s: float
    frame_path: str                       # JPEG written by this run (raw, un-annotated)
    detections: List[List[float]]         # [[x1, y1, x2, y2, conf], ...] in original pixels
    kind: str = "candidate"               # "candidate" (model found people) | "audit" (model found none)
    source_path: str = ""                 # original video / image path

    @property
    def det_count(self) -> int:
        return len(self.detections)

    @property
    def confidences(self) -> List[float]:
        return sorted((float(d[4]) for d in self.detections), reverse=True)

    @property
    def max_conf(self) -> float:
        return self.confidences[0] if self.detections else 0.0

    @property
    def mean_conf(self) -> float:
        return float(np.mean(self.confidences)) if self.detections else 0.0

    def to_dict(self) -> dict:
        return asdict(self)

    @classmethod
    def from_dict(cls, d: dict) -> "FrameRecord":
        return cls(**d)


@dataclass
class RunResult:
    run_id: str
    run_dir: Path
    records: List[FrameRecord]
    summary: dict
    all_frames: List[dict]                # lightweight per-frame detections for EVERY sampled frame


def cfg_to_jsonable(cfg) -> dict:
    out = {}
    for k, v in asdict(cfg).items():
        out[k] = str(v) if isinstance(v, Path) else (list(v) if isinstance(v, tuple) else v)
    return out


def release_memory(device: str) -> None:
    """Return cached buffers to the OS (important on long runs with unified memory)."""
    gc.collect()
    if device == "mps" and hasattr(torch, "mps"):
        try:
            torch.mps.empty_cache()
        except Exception:
            pass


# ----------------------------------------------------------------------------
# Model loading (+ MPS warm-up and automatic CPU fallback)
# ----------------------------------------------------------------------------
def load_model(cfg) -> YOLO:
    model = YOLO(cfg.weights)
    names = getattr(model, "names", {}) or {}
    if names.get(0) != "person":
        log.warning("Class 0 is '%s', not 'person' — these weights may not be COCO-style.", names.get(0))
    dummy = np.zeros((cfg.imgsz, cfg.imgsz, 3), dtype=np.uint8)
    try:
        model.predict(dummy, imgsz=cfg.imgsz, device=cfg.device, classes=list(cfg.classes),
                      conf=cfg.conf, verbose=False)
    except Exception as exc:
        if cfg.device == "cpu":
            raise
        log.warning("Inference on '%s' failed during warm-up (%s) → falling back to CPU.", cfg.device, exc)
        cfg.device = "cpu"
        model = YOLO(cfg.weights)
        model.predict(dummy, imgsz=cfg.imgsz, device="cpu", classes=list(cfg.classes),
                      conf=cfg.conf, verbose=False)
    log.info("Model '%s' ready on device '%s'.", cfg.weights, cfg.device)
    return model


# ----------------------------------------------------------------------------
# Detection helpers
# ----------------------------------------------------------------------------
def _predict_batch(model, images: List[np.ndarray], cfg, imgsz: int) -> List[np.ndarray]:
    """Run YOLO on a list of BGR images → list of (N, 5) arrays [x1, y1, x2, y2, conf]."""
    results = model.predict(
        source=images, imgsz=imgsz, conf=cfg.conf, iou=cfg.iou, classes=list(cfg.classes),
        max_det=cfg.max_det, agnostic_nms=cfg.agnostic_nms, device=cfg.device,
        # NOTE: no half=... argument on purpose: FP32 is the default, and newer Ultralytics warns that 'half' is deprecated.
        augment=False, verbose=False,
    )
    out = []
    for r in results:
        if r.boxes is None or len(r.boxes) == 0:
            out.append(np.zeros((0, 5), dtype=np.float32))
            continue
        xyxy = r.boxes.xyxy.cpu().numpy()
        conf = r.boxes.conf.cpu().numpy()[:, None]
        out.append(np.concatenate([xyxy, conf], axis=1).astype(np.float32))
    return out


def _axis_starts(length: int, tile: int, step: int) -> List[int]:
    if length <= tile:
        return [0]
    starts = list(range(0, length - tile + 1, step))
    if starts[-1] != length - tile:
        starts.append(length - tile)       # last tile is flush with the image edge
    return starts


def tile_origins(width: int, height: int, tile: int, overlap: float) -> List[Tuple[int, int, int, int]]:
    """Return (x, y, w, h) for every tile covering the image with the given overlap."""
    step = max(1, int(tile * (1.0 - overlap)))
    return [(x, y, min(tile, width), min(tile, height))
            for y in _axis_starts(height, tile, step) for x in _axis_starts(width, tile, step)]


def merge_detections(parts: List[np.ndarray], iou_thr: float, ios_thr: float) -> np.ndarray:
    """Greedy merge of detections from several passes.

    Suppresses a lower-score box if IoU > iou_thr  OR  if it is mostly *contained* in a
    higher-score box (intersection-over-smaller > ios_thr) — this removes the half-person
    fragments that appear where a person straddles two tiles.
    """
    parts = [p for p in parts if len(p)]
    if not parts:
        return np.zeros((0, 5), dtype=np.float32)
    d = np.concatenate(parts, axis=0)
    order = np.argsort(-d[:, 4])
    boxes = d[:, :4]
    areas = np.maximum(boxes[:, 2] - boxes[:, 0], 0) * np.maximum(boxes[:, 3] - boxes[:, 1], 0)
    suppressed = np.zeros(len(d), dtype=bool)
    keep = []
    for i in order:
        if suppressed[i]:
            continue
        keep.append(i)
        xx1 = np.maximum(boxes[i, 0], boxes[:, 0]); yy1 = np.maximum(boxes[i, 1], boxes[:, 1])
        xx2 = np.minimum(boxes[i, 2], boxes[:, 2]); yy2 = np.minimum(boxes[i, 3], boxes[:, 3])
        inter = np.maximum(xx2 - xx1, 0) * np.maximum(yy2 - yy1, 0)
        iou = inter / np.maximum(areas[i] + areas - inter, 1e-9)
        ios = inter / np.maximum(np.minimum(areas[i], areas), 1e-9)
        suppressed |= (iou > iou_thr) | (ios > ios_thr)
    return d[np.array(keep, dtype=int)]


def filter_boxes(d: np.ndarray, width: int, height: int, cfg) -> np.ndarray:
    """Clip to image, drop tiny noise boxes and implausibly huge ones."""
    if len(d) == 0:
        return d
    d = d.copy()
    d[:, [0, 2]] = np.clip(d[:, [0, 2]], 0, width - 1)
    d[:, [1, 3]] = np.clip(d[:, [1, 3]], 0, height - 1)
    bw, bh = d[:, 2] - d[:, 0], d[:, 3] - d[:, 1]
    keep = (bw >= cfg.min_box_px) & (bh >= cfg.min_box_px) & (bw * bh <= cfg.max_box_frac * width * height)
    return d[keep]


def detect_persons(model, frame_bgr: np.ndarray, cfg) -> np.ndarray:
    """All persons in one frame → (N, 5) float32 array [x1, y1, x2, y2, conf]."""
    h, w = frame_bgr.shape[:2]
    parts = _predict_batch(model, [frame_bgr], cfg, cfg.imgsz)            # full-frame pass
    if cfg.use_tiling and max(h, w) > cfg.tile_size * 1.2:                # tiling only helps big frames
        origins = tile_origins(w, h, cfg.tile_size, cfg.tile_overlap)
        for k in range(0, len(origins), cfg.tile_batch):
            chunk = origins[k:k + cfg.tile_batch]
            crops = [frame_bgr[y:y + th, x:x + tw] for (x, y, tw, th) in chunk]
            for (x, y, _, _), det in zip(chunk, _predict_batch(model, crops, cfg, cfg.tile_imgsz)):
                if len(det):
                    det = det.copy()
                    det[:, [0, 2]] += x                                    # tile → full-frame coordinates
                    det[:, [1, 3]] += y
                    parts.append(det)
    merged = merge_detections(parts, cfg.merge_iou, cfg.merge_ios)
    return filter_boxes(merged, w, h, cfg)


def annotate_frame(frame_bgr: np.ndarray, dets, max_side: Optional[int] = None,
                   draw_labels: bool = True) -> np.ndarray:
    """Return an RGB image with boxes drawn (colour = confidence band). Optionally downscaled first."""
    dets = np.asarray(dets, dtype=np.float32).reshape(-1, 5)
    h, w = frame_bgr.shape[:2]
    scale = 1.0
    if max_side and max(h, w) > max_side:
        scale = max_side / float(max(h, w))
        img = cv2.resize(frame_bgr, (max(1, int(w * scale)), max(1, int(h * scale))), interpolation=cv2.INTER_AREA)
    else:
        img = frame_bgr.copy()
    ih, iw = img.shape[:2]
    thick = max(1, int(round(max(ih, iw) / 700)))
    font = max(0.4, max(ih, iw) / 1800)
    for x1, y1, x2, y2, c in dets:
        p1 = (int(x1 * scale), int(y1 * scale)); p2 = (int(x2 * scale), int(y2 * scale))
        color = (80, 200, 60) if c >= 0.6 else (0, 165, 255) if c >= 0.4 else (60, 60, 230)   # BGR
        cv2.rectangle(img, p1, p2, color, thick)
        if draw_labels and len(dets) <= 40:
            label = f"{c:.2f}"
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font, 1)
            ty = max(p1[1], th + 4)
            cv2.rectangle(img, (p1[0], ty - th - 4), (p1[0] + tw + 4, ty), color, -1)
            cv2.putText(img, label, (p1[0] + 2, ty - 3), cv2.FONT_HERSHEY_SIMPLEX, font, (255, 255, 255), 1, cv2.LINE_AA)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


# ----------------------------------------------------------------------------
# End-to-end run
# ----------------------------------------------------------------------------
def process_source(cfg, model=None, show_progress: bool = True) -> RunResult:
    """Stream → detect → select candidates/audit frames → persist run artefacts."""
    meta, expected = describe_source(cfg)
    cfg.frames_dir.mkdir(parents=True, exist_ok=True)
    model = model or load_model(cfg)
    rng = random.Random(cfg.seed)
    stats = SamplerStats()
    gen = iter_source(cfg, meta, stats)

    records: List[FrameRecord] = []
    all_frames: List[dict] = []
    n_sampled = n_cand = n_neg = n_audit = total_dets = 0
    decode_s = infer_s = save_s = 0.0
    infer_times: List[float] = []
    first_ts = last_ts = None
    interrupted = False
    wall0 = time.perf_counter()
    bar = tqdm(total=expected, desc="Detecting persons", unit="frame", disable=not show_progress)
    try:
        while True:
            t0 = time.perf_counter()
            item = next(gen, None)                                  # decoding happens inside next()
            t1 = time.perf_counter()
            if item is None:
                break
            decode_s += t1 - t0
            try:
                dets = detect_persons(model, item.frame, cfg)
            except Exception as exc:
                if cfg.device != "mps":
                    raise
                log.warning("MPS inference failed (%s) → switching to CPU for the rest of the run.", exc)
                cfg.device = "cpu"
                model = load_model(cfg)
                dets = detect_persons(model, item.frame, cfg)
            t2 = time.perf_counter()
            infer_s += t2 - t1
            infer_times.append(t2 - t1)

            n_sampled += 1
            total_dets += len(dets)
            first_ts = item.timestamp_s if first_ts is None else first_ts
            last_ts = item.timestamp_s
            frame_id = f"{item.source_name}_f{item.index:08d}" if cfg.source_mode == "video" else item.source_name
            det_list = [[round(float(v), 2) for v in d[:4]] + [round(float(d[4]), 4)] for d in dets]
            all_frames.append({"frame_id": frame_id, "frame_index": item.index, "timestamp_s": round(item.timestamp_s, 3),
                               "source_path": item.source_path, "detections": det_list})

            kind = None
            if len(dets) >= cfg.min_detections:
                kind, n_cand = "candidate", n_cand + 1
            else:
                n_neg += 1
                if rng.random() < cfg.audit_fraction:
                    kind, n_audit = "audit", n_audit + 1
            if kind:
                t3 = time.perf_counter()
                path = cfg.frames_dir / f"{frame_id}.jpg"
                save_frame_jpeg(item.frame, path, cfg.jpeg_quality)
                save_s += time.perf_counter() - t3
                records.append(FrameRecord(frame_id, item.index, round(item.timestamp_s, 3), str(path.resolve()),
                                           det_list, kind, item.source_path))
            del item, dets
            bar.update(1)
            bar.set_postfix(candidates=n_cand, audit=n_audit)
            if n_sampled % 25 == 0:
                release_memory(cfg.device)
    except KeyboardInterrupt:
        interrupted = True
        log.warning("Interrupted by user — saving partial results.")
    finally:
        bar.close()
        gen.close()
        release_memory(cfg.device)
    wall = time.perf_counter() - wall0

    covered = max((last_ts or 0) - (first_ts or 0), cfg.sample_interval_s) if cfg.source_mode == "video" else float(n_sampled)
    arr = np.array(infer_times) if infer_times else np.array([0.0])
    summary = {
        "run_id": cfg.run_id, "created_at": datetime.now().astimezone().isoformat(timespec="seconds"),
        "interrupted": interrupted, "source_mode": cfg.source_mode, "data_source_label": cfg.data_source_label,
        "video": meta.as_dict() if meta else None,
        "device": cfg.device, "weights": cfg.weights, "torch": torch.__version__, "ultralytics": ultralytics.__version__,
        "config": cfg_to_jsonable(cfg),
        "counts": {"sampled_frames": n_sampled, "candidate_frames": n_cand, "negative_frames": n_neg,
                   "audit_frames": n_audit, "total_detections": total_dets},
        "timing": {"wall_s": wall, "decode_s": decode_s, "inference_s": infer_s, "save_s": save_s,
                   "mean_infer_s": float(arr.mean()), "p50_infer_s": float(np.percentile(arr, 50)),
                   "p95_infer_s": float(np.percentile(arr, 95)), "source_seconds_covered": float(covered)},
        "sampler": asdict(stats),
    }
    (cfg.run_dir / "candidates.json").write_text(json.dumps({"records": [r.to_dict() for r in records]}))
    (cfg.run_dir / "all_frames.json").write_text(json.dumps({"frames": all_frames}))
    (cfg.run_dir / "run_summary.json").write_text(json.dumps(summary, indent=2, default=str))
    return RunResult(cfg.run_id, cfg.run_dir, records, summary, all_frames)


def load_run_result(run_dir) -> RunResult:
    """Re-open a finished run (resume reviewing without re-running inference)."""
    run_dir = Path(run_dir).expanduser()
    summary = json.loads((run_dir / "run_summary.json").read_text())
    recs = [FrameRecord.from_dict(d) for d in json.loads((run_dir / "candidates.json").read_text())["records"]]
    allf_path = run_dir / "all_frames.json"
    allf = json.loads(allf_path.read_text())["frames"] if allf_path.exists() else []
    return RunResult(summary["run_id"], run_dir, recs, summary, allf)


def print_run_summary(res: RunResult) -> None:
    c, t = res.summary["counts"], res.summary["timing"]
    n = max(c["sampled_frames"], 1)
    print(f"✔ Run {res.run_id} on {res.summary['device'].upper()} | source: {res.summary['data_source_label']}")
    print(f"  frames analysed : {c['sampled_frames']}  | candidates: {c['candidate_frames']}  "
          f"| empty frames: {c['negative_frames']} (audit sample: {c['audit_frames']})")
    print(f"  detections      : {c['total_detections']}")
    print(f"  speed           : {t['inference_s'] / n:.3f} s/frame inference → {n / max(t['inference_s'], 1e-9):.2f} FPS "
          f"| end-to-end {n / max(t['wall_s'], 1e-9):.2f} FPS")
    if res.summary["sampler"]["corrupt_frames_skipped"] or res.summary["sampler"]["truncated"]:
        print(f"  ⚠ sampler: {res.summary['sampler']}")
    print(f"  artefacts       : {res.run_dir}")


# ----------------------------------------------------------------------------
# Load model + (if the input exists) run the pipeline
# ----------------------------------------------------------------------------
MODEL = load_model(CFG)
RESULT: Optional[RunResult] = None
_input_path = CFG.video_path if CFG.source_mode == "video" else CFG.images_dir
if Path(_input_path).exists():
    try:
        RESULT = process_source(CFG, MODEL)
        print_run_summary(RESULT)
    except (VideoError, ValueError) as exc:
        print(f"✖ Cannot process input: {exc}")
else:
    print(f"ℹ Input not found: {_input_path}\n  Set CFG.source_mode / CFG.video_path / CFG.images_dir, then re-run this cell.")


---
## Cell 4 — Human review dashboard

| Key | Action |
|-----|--------|
| `c` | **Confirm** — person present |
| `r` | **Reject** — false alarm |
| `f` | **Flag** — needs second look |
| `n` | Next frame |
| `p` | Previous frame |
| `u` | Undo / clear decision |

Decisions are held in memory until Cell 5 binds the audit trail. Re-run Cell 5 after reviewing to refresh the CSV.


In [ ]:
# ==============================================================================
# AAROHAN | CELL 4 — Human-in-the-Loop Review Dashboard (ipywidgets + PIL)
# ==============================================================================
# • One flicker-free ipywidgets.Image is updated in place (JPEG bytes) — no matplotlib redraws.
# • Overview image + zoomed crops of the most confident detections (aerial persons are tiny).
# • Buttons: Confirm Person (TP) | Reject False Alarm (FP) | Needs Review / Flag (uncertain)
# • Keyboard:  c = confirm   r = reject   f = flag   n = next   p = previous   u = undo
#   (shortcuts need JupyterLab / Notebook 7 / classic Notebook; VS Code → use the buttons)
# • Audit frames (model found nothing) use the same buttons with a different meaning —
#   a banner explains it — so Cell 6 can estimate missed detections (false negatives).
# • Every decision is pushed to registered callbacks → Cell 5 writes the audit trail.
# ------------------------------------------------------------------------------
DECISION_CONFIRMED = "confirmed"
DECISION_REJECTED = "rejected"
DECISION_FLAGGED = "flagged"
DECISION_PENDING = "pending"
VALID_DECISIONS = (DECISION_CONFIRMED, DECISION_REJECTED, DECISION_FLAGGED)


def now_iso() -> str:
    """Local time with UTC offset, e.g. 2026-10-24T14:03:22+05:30"""
    return datetime.now().astimezone().isoformat(timespec="seconds")


def _jpeg_bytes(rgb: np.ndarray, quality: int = 88) -> bytes:
    buf = io.BytesIO()
    Image.fromarray(rgb).save(buf, format="JPEG", quality=quality)
    return buf.getvalue()


def _placeholder_bytes(text: str, size=(900, 500)) -> bytes:
    img = Image.new("RGB", size, (38, 41, 46))
    ImageDraw.Draw(img).text((20, size[1] // 2 - 6), text, fill=(230, 230, 230))
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return buf.getvalue()


# JavaScript that maps keys → button clicks. It runs in the notebook page itself.
_KEYBOARD_JS = r"""
(function () {
  if (window.__aarohanKeyHandler) {
    document.removeEventListener('keydown', window.__aarohanKeyHandler, true);
  }
  const MAP = {c: 'aarohan-btn-confirm', r: 'aarohan-btn-reject', f: 'aarohan-btn-flag',
               n: 'aarohan-btn-next', p: 'aarohan-btn-prev', u: 'aarohan-btn-undo'};
  window.__aarohanKeyHandler = function (ev) {
    if (ev.ctrlKey || ev.metaKey || ev.altKey || ev.repeat) return;
    const cls = MAP[(ev.key || '').toLowerCase()];
    if (!cls) return;
    const t = ev.target;
    // never hijack typing in text boxes or code cells
    if (t && t.closest && t.closest('input, textarea, select, [contenteditable="true"], .cm-editor, .CodeMirror')) return;
    const toggle = document.querySelector('.aarohan-keys-toggle input');
    if (toggle && !toggle.checked) return;
    const nodes = Array.from(document.querySelectorAll('.' + cls)).filter(e => e.offsetParent !== null);
    if (!nodes.length) return;
    const el = nodes[nodes.length - 1];
    ev.preventDefault(); ev.stopPropagation(); ev.stopImmediatePropagation();
    (el.tagName === 'BUTTON' ? el : (el.querySelector('button') || el)).click();
  };
  document.addEventListener('keydown', window.__aarohanKeyHandler, true);
})();
"""


class ReviewDashboard:
    SORT_OPTIONS = [("Video time (earliest first)", "time"),
                    ("Confidence — lowest first (most uncertain)", "conf_asc"),
                    ("Confidence — highest first", "conf_desc"),
                    ("Detection count — most first", "count_desc")]
    ZOOM_TILE = 150          # px per thumbnail
    ZOOM_COLS = 4
    ZOOM_MAX = 8             # thumbnails shown per frame

    def __init__(self, result: "RunResult", max_display_px: int = 1100, reviewer_id: Optional[str] = None):
        self.records: List[FrameRecord] = list(result.records)
        self.summary = result.summary
        self.max_display_px = int(max_display_px)
        self.decisions: Dict[str, dict] = {}                 # frame_id -> latest decision entry
        self.on_decision: List[Callable[[FrameRecord, dict], None]] = []
        self.export_handler: Optional[Callable[[], object]] = None
        self._cache = LRUCache(8)                            # frame_id -> (overview_jpeg, zoom_jpeg)
        self.order: List[int] = []
        self.pos = 0
        self._default_reviewer = reviewer_id or getpass.getuser()
        self._build_ui()
        self._wire()
        self._rebuild_queue(keep_current=False)
        self._render()

    # ------------------------------------------------------------------ UI
    def _button(self, label, style, icon, tip, css):
        b = widgets.Button(description=label, button_style=style, icon=icon, tooltip=tip,
                           layout=widgets.Layout(flex="1 1 0%", width="auto", height="46px"))
        b.add_class(css)
        return b

    def _build_ui(self):
        L = widgets.Layout
        self.title = widgets.HTML("<div style='font-size:18px;font-weight:600'>AAROHAN — Human-in-the-Loop Review</div>"
                                  f"<div style='color:#57606a;font-size:12px'>{html.escape(str(self.summary.get('data_source_label', '')))}"
                                  f" · run {html.escape(str(self.summary.get('run_id', '')))}</div>")
        self.progress = widgets.IntProgress(value=0, min=0, max=max(1, len(self.records)), bar_style="info",
                                            layout=L(width="320px"))
        self.progress_txt = widgets.HTML()
        self.sort_dd = widgets.Dropdown(options=self.SORT_OPTIONS, value="time", description="Order:",
                                        layout=L(width="340px"))
        self.pending_only = widgets.Checkbox(value=True, description="Pending only", indent=False, layout=L(width="130px"))
        self.auto_adv = widgets.Checkbox(value=True, description="Auto-advance", indent=False, layout=L(width="130px"))
        self.keys_on = widgets.Checkbox(value=True, description="Keyboard shortcuts", indent=False, layout=L(width="170px"))
        self.keys_on.add_class("aarohan-keys-toggle")

        self.img_main = widgets.Image(format="jpeg", layout=L(width="100%"))
        self.meta = widgets.HTML()
        self.zoom_title = widgets.HTML("<b style='font-size:12px'>Top detections (zoom, from full-resolution frame)</b>")
        self.img_zoom = widgets.Image(format="jpeg", layout=L(width="100%", max_width="640px"))
        self.reviewer = widgets.Text(value=self._default_reviewer, description="Reviewer:", layout=L(width="100%"))
        self.note = widgets.Text(placeholder="Optional note for this frame", description="Note:", layout=L(width="100%"))

        self.btn_confirm = self._button("Confirm Person  [C]", "success", "check", "True Positive — a person is visible (key: c)", "aarohan-btn-confirm")
        self.btn_reject = self._button("Reject False Alarm  [R]", "danger", "times", "False Positive — no person here (key: r)", "aarohan-btn-reject")
        self.btn_flag = self._button("Needs Review / Flag  [F]", "warning", "flag", "Uncertain — escalate (key: f)", "aarohan-btn-flag")
        self.btn_prev = self._button("◀ Previous [P]", "", "", "Previous frame (key: p)", "aarohan-btn-prev")
        self.btn_next = self._button("Next [N] ▶", "info", "", "Skip to next frame without a decision (key: n)", "aarohan-btn-next")
        self.btn_undo = self._button("Clear decision [U]", "", "undo", "Remove the decision for this frame (key: u)", "aarohan-btn-undo")
        self.btn_export = self._button("Export CSV now", "primary", "save", "Write detections_log.csv", "aarohan-btn-export")
        self.status = widgets.HTML()
        hint = widgets.HTML("<span style='color:#57606a;font-size:12px'>Keys: <b>c</b> confirm · <b>r</b> reject · "
                            "<b>f</b> flag · <b>n</b> next · <b>p</b> previous · <b>u</b> clear. "
                            "Keys are ignored while typing in a text box.</span>")

        side = widgets.VBox([self.meta, self.zoom_title, self.img_zoom, self.reviewer, self.note],
                            layout=L(flex="1 1 36%", padding="0 0 0 12px", min_width="300px"))
        main = widgets.HBox([widgets.VBox([self.img_main], layout=L(flex="1 1 64%")), side],
                            layout=L(width="100%", align_items="flex-start"))
        self.root = widgets.VBox([
            self.title,
            widgets.HBox([self.progress, self.progress_txt], layout=L(align_items="center")),
            widgets.HBox([self.sort_dd, self.pending_only, self.auto_adv, self.keys_on], layout=L(flex_flow="row wrap")),
            main,
            widgets.HBox([self.btn_confirm, self.btn_reject, self.btn_flag], layout=L(width="100%", margin="8px 0 4px 0")),
            widgets.HBox([self.btn_prev, self.btn_next, self.btn_undo, self.btn_export], layout=L(width="100%")),
            self.status, hint,
        ], layout=L(width="100%", max_width="1500px", border="1px solid #d0d7de", padding="12px"))

    def _wire(self):
        self.btn_confirm.on_click(lambda _b: self._decide(DECISION_CONFIRMED))
        self.btn_reject.on_click(lambda _b: self._decide(DECISION_REJECTED))
        self.btn_flag.on_click(lambda _b: self._decide(DECISION_FLAGGED))
        self.btn_prev.on_click(lambda _b: self._go(-1))
        self.btn_next.on_click(lambda _b: self._go(+1))
        self.btn_undo.on_click(lambda _b: self._clear())
        self.btn_export.on_click(lambda _b: self._export())
        self.sort_dd.observe(self._on_options, names="value")
        self.pending_only.observe(self._on_options, names="value")

    # --------------------------------------------------------------- queue
    def _current(self) -> Optional[FrameRecord]:
        return self.records[self.order[self.pos]] if self.order else None

    def _rebuild_queue(self, keep_current: bool = True):
        cur = self._current().frame_id if (keep_current and self.order) else None
        idxs = list(range(len(self.records)))
        if self.pending_only.value:
            idxs = [i for i in idxs if self.records[i].frame_id not in self.decisions]
        key = self.sort_dd.value
        rec = self.records
        if key == "conf_asc":
            idxs.sort(key=lambda i: (rec[i].kind == "audit", rec[i].max_conf, rec[i].frame_index))
        elif key == "conf_desc":
            idxs.sort(key=lambda i: (rec[i].kind == "audit", -rec[i].max_conf, rec[i].frame_index))
        elif key == "count_desc":
            idxs.sort(key=lambda i: (-rec[i].det_count, rec[i].frame_index))
        else:
            idxs.sort(key=lambda i: rec[i].frame_index)
        self.order = idxs
        self.pos = 0
        if cur is not None:
            for p, i in enumerate(self.order):
                if rec[i].frame_id == cur:
                    self.pos = p
                    break

    def refresh(self):
        """Public: rebuild the queue and redraw (used after decisions are restored from disk)."""
        self._rebuild_queue(keep_current=False)
        self._render()

    def _on_options(self, _change):
        self._rebuild_queue(keep_current=True)
        self._render()

    # --------------------------------------------------------------- actions
    def _notify(self, rec: FrameRecord, entry: dict):
        for cb in list(self.on_decision):
            try:
                cb(rec, entry)
            except Exception as exc:                        # a logging failure must never freeze the UI
                log.exception("on_decision callback failed")
                self._set_status(f"⚠ Audit-trail callback failed: {html.escape(str(exc))}", "#9a6700")

    def _decide(self, decision: str):
        rec = self._current()
        if rec is None:
            return
        entry = {"decision": decision, "reviewer_id": (self.reviewer.value or "").strip() or "unknown",
                 "note": (self.note.value or "").strip(), "review_timestamp": now_iso()}
        self.decisions[rec.frame_id] = entry
        self._notify(rec, entry)
        at_end = self.pos >= len(self.order) - 1
        if self.auto_adv.value and not at_end:
            self.pos += 1
            self._set_status(f"Saved <b>{decision}</b> for {html.escape(rec.frame_id)} → next frame.", "#1a7f37")
        elif at_end:
            pending = sum(1 for r in self.records if r.frame_id not in self.decisions)
            self._set_status(f"Saved <b>{decision}</b>. End of queue — {pending} frame(s) still pending overall.", "#1a7f37")
        else:
            self._set_status(f"Saved <b>{decision}</b> for {html.escape(rec.frame_id)}.", "#1a7f37")
        self._render()

    def _clear(self):
        rec = self._current()
        if rec is None or rec.frame_id not in self.decisions:
            return
        del self.decisions[rec.frame_id]
        self._notify(rec, {"decision": DECISION_PENDING, "reviewer_id": (self.reviewer.value or "").strip() or "unknown",
                           "note": "", "review_timestamp": now_iso()})
        self._set_status(f"Cleared decision for {html.escape(rec.frame_id)}.", "#57606a")
        self._render()

    def _go(self, delta: int):
        if not self.order:
            return
        new = min(max(self.pos + delta, 0), len(self.order) - 1)
        if new == self.pos:
            self._set_status("Already at the %s of the queue." % ("end" if delta > 0 else "start"), "#57606a")
            return
        self.pos = new
        self._render()

    def _export(self):
        if self.export_handler is None:
            self._set_status("Run Cell 5 first — it connects the CSV exporter to this dashboard.", "#9a6700")
            return
        try:
            self.export_handler()
            self._set_status("detections_log.csv written.", "#1a7f37")
        except Exception as exc:
            self._set_status(f"Export failed: {html.escape(str(exc))}", "#cf222e")

    def _set_status(self, text: str, color: str = "#57606a"):
        self.status.value = f"<div style='color:{color};font-size:13px;min-height:20px'>{text}</div>"

    # --------------------------------------------------------------- rendering
    def _make_zoom(self, frame: np.ndarray, dets: np.ndarray) -> bytes:
        if len(dets) == 0:
            return _placeholder_bytes("Audit frame: model found no person", (600, 150))
        top = dets[np.argsort(-dets[:, 4])][: self.ZOOM_MAX]
        h, w = frame.shape[:2]
        T, cols = self.ZOOM_TILE, self.ZOOM_COLS
        rows = int(math.ceil(len(top) / cols))
        sheet = Image.new("RGB", (cols * T, rows * T), (30, 30, 30))
        for k, (x1, y1, x2, y2, c) in enumerate(top):
            side = max(max(x2 - x1, y2 - y1) * 3.0, 64.0)           # context around the person
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            ax1, ay1 = int(max(0, cx - side / 2)), int(max(0, cy - side / 2))
            ax2, ay2 = int(min(w, cx + side / 2)), int(min(h, cy + side / 2))
            crop = frame[ay1:ay2, ax1:ax2]
            if crop.size == 0:
                continue
            tile = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)).resize((T, T), Image.LANCZOS)
            d = ImageDraw.Draw(tile)
            sx, sy = T / max(ax2 - ax1, 1), T / max(ay2 - ay1, 1)
            d.rectangle([(x1 - ax1) * sx, (y1 - ay1) * sy, (x2 - ax1) * sx, (y2 - ay1) * sy], outline=(60, 255, 90), width=2)
            d.rectangle([0, 0, 52, 14], fill=(0, 0, 0))
            d.text((3, 1), f"#{k + 1} {c:.2f}", fill=(255, 255, 255))
            sheet.paste(tile, ((k % cols) * T, (k // cols) * T))
        return _jpeg_bytes(np.asarray(sheet), 88)

    def _prepare(self, rec: FrameRecord) -> Tuple[bytes, bytes]:
        cached = self._cache.get(rec.frame_id)
        if cached is not None:
            return cached
        frame = load_frame(rec.frame_path)
        if frame is None:
            out = (_placeholder_bytes(f"Image missing or unreadable: {rec.frame_path}"), _placeholder_bytes("n/a", (600, 150)))
        else:
            dets = np.asarray(rec.detections, dtype=np.float32).reshape(-1, 5)
            out = (_jpeg_bytes(annotate_frame(frame, dets, max_side=self.max_display_px), 88), self._make_zoom(frame, dets))
        self._cache.put(rec.frame_id, out)
        return out

    def _badge(self, text: str, bg: str) -> str:
        return (f"<span style='background:{bg};color:#fff;padding:2px 8px;border-radius:10px;font-size:12px'>"
                f"{html.escape(text)}</span>")

    def _meta_html(self, rec: FrameRecord) -> str:
        entry = self.decisions.get(rec.frame_id)
        colors = {DECISION_CONFIRMED: "#1a7f37", DECISION_REJECTED: "#cf222e", DECISION_FLAGGED: "#bf8700"}
        dec = self._badge(entry["decision"].upper(), colors[entry["decision"]]) if entry else self._badge("PENDING", "#6e7781")
        kind = self._badge("MODEL CANDIDATE", "#0969da") if rec.kind == "candidate" else self._badge("AUDIT FRAME", "#8250df")
        confs = ", ".join(f"{c:.2f}" for c in rec.confidences[:10]) or "—"
        extra = "…" if rec.det_count > 10 else ""
        banner = ""
        if rec.kind == "audit":
            banner = ("<div style='background:#f6f0ff;border:1px solid #d8b9ff;padding:6px;margin:6px 0;font-size:12px'>"
                      "<b>Audit frame</b> — the model found <u>no</u> person. "
                      "<b>Confirm Person</b> = a person IS visible (missed detection). "
                      "<b>Reject</b> = frame is genuinely empty.</div>")
        ts = f"{format_hms(rec.timestamp_s)}" if self.summary.get("source_mode") == "video" else f"image #{int(rec.timestamp_s)}"
        return (f"<div style='font-size:13px;line-height:1.55'>{kind} {dec}{banner}"
                f"<table style='margin-top:6px;font-size:13px'>"
                f"<tr><td style='color:#57606a;padding-right:10px'>Frame</td><td><code>{html.escape(rec.frame_id)}</code></td></tr>"
                f"<tr><td style='color:#57606a'>Timestamp</td><td>{ts} (frame {rec.frame_index})</td></tr>"
                f"<tr><td style='color:#57606a'>Detections</td><td><b>{rec.det_count}</b></td></tr>"
                f"<tr><td style='color:#57606a'>Max / mean conf.</td><td><b>{rec.max_conf:.2f}</b> / {rec.mean_conf:.2f}</td></tr>"
                f"<tr><td style='color:#57606a'>Scores</td><td>{confs}{extra}</td></tr>"
                f"<tr><td style='color:#57606a'>Queue</td><td>{self.pos + 1} of {len(self.order)}</td></tr>"
                f"</table></div>")

    def _update_progress(self):
        n = len(self.records)
        done = len(self.decisions)
        c = sum(1 for d in self.decisions.values() if d["decision"] == DECISION_CONFIRMED)
        r = sum(1 for d in self.decisions.values() if d["decision"] == DECISION_REJECTED)
        f = sum(1 for d in self.decisions.values() if d["decision"] == DECISION_FLAGGED)
        self.progress.max = max(1, n)
        self.progress.value = done
        self.progress.bar_style = "success" if (n and done == n) else "info"
        self.progress_txt.value = (f"<span style='margin-left:10px;font-size:13px'>Reviewed <b>{done}/{n}</b> · "
                                   f"✅ {c} · ❌ {r} · 🚩 {f}</span>")

    def _render(self):
        self._update_progress()
        rec = self._current()
        no_rec = rec is None
        for b in (self.btn_confirm, self.btn_reject, self.btn_flag, self.btn_prev, self.btn_next, self.btn_undo):
            b.disabled = no_rec
        if no_rec:
            self.img_main.value = _placeholder_bytes("Nothing to review in the current queue.")
            self.img_zoom.value = _placeholder_bytes("", (600, 150))
            msg = ("No frames in the queue. " + ("Everything is reviewed — export the CSV (Cell 5)."
                   if self.records else "The model produced no candidate frames — lower CFG.conf or check the input."))
            self.meta.value = f"<div style='font-size:14px;padding:8px'>{msg}</div>"
            return
        try:
            main, zoom = self._prepare(rec)
            self.img_main.value, self.img_zoom.value = main, zoom
        except Exception as exc:
            log.exception("render failed")
            self.img_main.value = _placeholder_bytes(f"Render error: {exc}")
        self.meta.value = self._meta_html(rec)
        self.note.value = self.decisions.get(rec.frame_id, {}).get("note", "")

    def show(self):
        """Display the dashboard and install the keyboard shortcut handler."""
        display(self.root)
        display(Javascript(_KEYBOARD_JS))


# ----------------------------------------------------------------------------
# Launch
# ----------------------------------------------------------------------------
_res = globals().get("RESULT")
dashboard: Optional[ReviewDashboard] = None
if _res is None:
    print("ℹ No RESULT yet — run Cell 3 successfully first (or: RESULT = load_run_result(<run folder>)).")
elif not _res.records:
    print("ℹ The run produced no review frames (no detections and no audit sample). Try a lower CFG.conf.")
else:
    dashboard = ReviewDashboard(_res, max_display_px=CFG.display_max_px)
    dashboard.show()


---
## Cell 5 — CSV exporter & audit trail

Writes:
- `detections_log.csv` — one row per reviewed (or all) frame with decision
- `review_events.csv` — append-only event history (who/when/what)

Binds to the dashboard so later decisions are captured automatically. Re-run any time to refresh exports.


In [ ]:
# ==============================================================================
# AAROHAN | CELL 5 — CSV Exporter & Audit Trail Manager
# ==============================================================================
# Two files are maintained inside the run folder:
#   detections_log.csv  — LATEST decision per reviewed frame (atomically rewritten after every click)
#   review_events.csv   — APPEND-ONLY history of every decision / change / clear (tamper-evident trail)
#
# Binding is order-independent: if you reviewed frames BEFORE running this cell, those
# decisions are back-filled; if a log from an earlier session exists, it is restored so
# you can resume where you stopped.
# ------------------------------------------------------------------------------
LOG_COLUMNS = [
    "run_id", "frame_id", "frame_kind", "data_source", "source_path", "frame_index",
    "video_timestamp_s", "video_timestamp_hms", "frame_path", "detection_count",
    "max_confidence", "mean_confidence", "confidence_scores",
    "decision", "reviewer_id", "note", "review_timestamp",
]
EVENT_COLUMNS = LOG_COLUMNS + ["event"]


class AuditTrail:
    def __init__(self, result: "RunResult", log_path=None, events_path=None):
        self.result = result
        self.run_id = result.run_id
        self.by_id: Dict[str, FrameRecord] = {r.frame_id: r for r in result.records}
        self.log_path = Path(log_path or (Path(result.run_dir) / "detections_log.csv"))
        self.events_path = Path(events_path or (Path(result.run_dir) / "review_events.csv"))
        self.data_source = str(result.summary.get("data_source_label", ""))
        self._state: Dict[str, dict] = {}                      # frame_id -> latest decision entry
        self.log_path.parent.mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------ rows
    def _row(self, rec: FrameRecord, entry: dict) -> dict:
        return {
            "run_id": self.run_id, "frame_id": rec.frame_id, "frame_kind": rec.kind,
            "data_source": self.data_source, "source_path": rec.source_path, "frame_index": rec.frame_index,
            "video_timestamp_s": f"{rec.timestamp_s:.3f}", "video_timestamp_hms": format_hms(rec.timestamp_s),
            "frame_path": rec.frame_path, "detection_count": rec.det_count,
            "max_confidence": f"{rec.max_conf:.4f}", "mean_confidence": f"{rec.mean_conf:.4f}",
            "confidence_scores": ";".join(f"{c:.4f}" for c in rec.confidences),
            "decision": entry.get("decision", DECISION_PENDING), "reviewer_id": entry.get("reviewer_id", ""),
            "note": entry.get("note", ""), "review_timestamp": entry.get("review_timestamp", ""),
        }

    # ------------------------------------------------------------ events (append-only)
    def _append_event(self, rec: FrameRecord, entry: dict, event: str):
        row = self._row(rec, entry)
        row["event"] = event
        new_file = not self.events_path.exists()
        with open(self.events_path, "a", newline="", encoding="utf-8") as fh:
            writer = csv.DictWriter(fh, fieldnames=EVENT_COLUMNS)
            if new_file:
                writer.writeheader()
            writer.writerow(row)
            fh.flush()
            os.fsync(fh.fileno())                              # survive a kernel crash / power loss

    def _apply(self, rec: FrameRecord, entry: dict):
        """Update in-memory state and append to the event history (no CSV rewrite)."""
        previous = self._state.get(rec.frame_id)
        if entry["decision"] == DECISION_PENDING:
            self._state.pop(rec.frame_id, None)
            event = "cleared"
        else:
            self._state[rec.frame_id] = dict(entry)
            event = "changed" if previous else "decided"
        self._append_event(rec, entry, event)

    def _on_decision(self, rec: FrameRecord, entry: dict):
        self._apply(rec, entry)
        self.export()

    _on_decision._aarohan_audit = True

    # ------------------------------------------------------------ export (atomic)
    def export(self, include_pending: bool = False) -> Path:
        rows = []
        for rec in sorted(self.result.records, key=lambda r: (r.frame_index, r.frame_id)):
            entry = self._state.get(rec.frame_id)
            if entry is None and not include_pending:
                continue
            rows.append(self._row(rec, entry or {"decision": DECISION_PENDING}))
        tmp = self.log_path.with_suffix(".csv.tmp")
        pd.DataFrame(rows, columns=LOG_COLUMNS).to_csv(tmp, index=False)
        os.replace(tmp, self.log_path)                         # atomic: never leaves a half-written file
        return self.log_path

    def to_dataframe(self) -> pd.DataFrame:
        if not self.log_path.exists():
            self.export()
        return pd.read_csv(self.log_path, dtype=str, keep_default_na=False)

    # ------------------------------------------------------------ resume
    def _restore_from_csv(self) -> int:
        if not self.log_path.exists():
            return 0
        try:
            df = pd.read_csv(self.log_path, dtype=str, keep_default_na=False)
        except Exception as exc:
            log.warning("Could not read existing log (%s) — starting clean.", exc)
            return 0
        n = 0
        for _, row in df.iterrows():
            if row.get("run_id") != self.run_id or row.get("frame_id") not in self.by_id:
                continue
            if row.get("decision") not in VALID_DECISIONS:
                continue
            self._state[row["frame_id"]] = {"decision": row["decision"], "reviewer_id": row.get("reviewer_id", ""),
                                            "note": row.get("note", ""), "review_timestamp": row.get("review_timestamp", "")}
            n += 1
        return n

    def bind(self, dash: "ReviewDashboard", restore: bool = True) -> None:
        """Connect to the dashboard: restore old decisions, back-fill current ones, autosave from now on."""
        restored = self._restore_from_csv() if restore else 0
        for fid, entry in dash.decisions.items():              # decisions made before this cell ran win
            rec = self.by_id.get(fid)
            if rec is not None and self._state.get(fid, {}).get("review_timestamp") != entry["review_timestamp"]:
                self._apply(rec, entry)
        for fid, entry in self._state.items():                 # push restored decisions into the UI
            dash.decisions.setdefault(fid, dict(entry))
        dash.on_decision = [cb for cb in dash.on_decision if not getattr(cb, "_aarohan_audit", False)]
        dash.on_decision.append(self._on_decision)
        dash.export_handler = self.export
        dash.refresh()
        path = self.export()
        print(f"✔ Audit trail bound. Restored {restored} earlier decision(s); {len(self._state)} on record.")
        print(f"  latest decisions → {path}\n  full event history → {self.events_path}")


def export_detections_log(audit: AuditTrail, include_pending: bool = False) -> pd.DataFrame:
    """Convenience: write the CSV and return it as a DataFrame."""
    path = audit.export(include_pending=include_pending)
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    print(f"✔ {len(df)} row(s) written to {path}")
    return df


_res = globals().get("RESULT")
audit: Optional[AuditTrail] = None
if _res is None or not _res.records:
    print("ℹ Nothing to log yet — run Cells 3 and 4 first.")
else:
    audit = AuditTrail(_res)
    _dash = globals().get("dashboard")
    if _dash is not None:
        audit.bind(_dash)
    else:
        audit.export()
        print(f"✔ Audit trail ready (no dashboard bound). Log: {audit.log_path}")


---
## Cell 6 — Performance metrics

**A) Reviewer-based** (any footage, including real incidents without labels)
- Precision from reviewed candidates
- Recall & FPR *estimated* from the audit sample (Wilson 95% CI)

**B) Ground-truth** (when `CFG.labels_dir` points at HERIDAL / SARD / …)
- Detection-level and frame-level P/R/F1 @ IoU ≥ 0.5
- Reviewer-vs-truth agreement

**C) Ground-truth only** — if no reviewer decisions yet but labels exist, runs the objective benchmark automatically.

Frame-level rule: a frame is positive if it holds ≥ 1 person.


In [ ]:
# ==============================================================================
# AAROHAN | CELL 6 — Performance Metrics: Precision / Recall / FPR / Speed
# ==============================================================================
# Two independent evaluations:
#   A) REVIEWER-BASED  (works on any footage, incl. real incidents with no labels)
#        precision      = confirmed / (confirmed + rejected)           on candidate frames
#        recall, FPR    = ESTIMATED from the randomly sampled audit frames (see Cell 3)
#   B) GROUND-TRUTH    (only when CFG.labels_dir points at a labelled dataset such as
#        HERIDAL / SARD / WiSARD): objective detection-level and frame-level metrics,
#        plus how well the human reviewers agreed with the ground truth.
# Frame-level metrics: a frame counts as "positive" if it contains >= 1 person.
# ------------------------------------------------------------------------------

def wilson_interval(k: int, n: int, z: float = 1.96) -> Tuple[Optional[float], Optional[float]]:
    """95 % Wilson score interval for a proportion k/n."""
    if n <= 0:
        return None, None
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return max(0.0, centre - half), min(1.0, centre + half)


def _ratio(num: float, den: float) -> Optional[float]:
    return float(num / den) if den and den > 0 else None


# ----------------------------------------------------------------------------
# A) Reviewer-based metrics
# ----------------------------------------------------------------------------
def compute_review_metrics(log_df: pd.DataFrame, summary: dict) -> dict:
    d = log_df[log_df["decision"].isin(VALID_DECISIONS)].copy()
    cand, aud = d[d["frame_kind"] == "candidate"], d[d["frame_kind"] == "audit"]
    count = lambda df, label: int((df["decision"] == label).sum())

    tp, fp, fl_c = count(cand, DECISION_CONFIRMED), count(cand, DECISION_REJECTED), count(cand, DECISION_FLAGGED)
    fn_a, tn_a, fl_a = count(aud, DECISION_CONFIRMED), count(aud, DECISION_REJECTED), count(aud, DECISION_FLAGGED)
    c = summary["counts"]
    cand_total, neg_total = c["candidate_frames"], c["negative_frames"]

    # Partial review? scale decided counts up to the full population (flagged are excluded, reported separately).
    cand_scale = _ratio(cand_total, tp + fp) or 1.0
    aud_scale = _ratio(neg_total, fn_a + tn_a)
    tp_e, fp_e = tp * cand_scale, fp * cand_scale
    fn_e = fn_a * aud_scale if aud_scale else None
    tn_e = tn_a * aud_scale if aud_scale else None

    precision = _ratio(tp, tp + fp)
    p_lo, p_hi = wilson_interval(tp, tp + fp)
    recall = _ratio(tp_e, tp_e + fn_e) if fn_e is not None else None
    fpr = _ratio(fp_e, fp_e + tn_e) if tn_e is not None else None
    f1 = _ratio(2 * precision * recall, precision + recall) if (precision and recall) else None
    miss_lo, miss_hi = wilson_interval(fn_a, fn_a + tn_a)

    t = summary["timing"]
    n = max(c["sampled_frames"], 1)
    return {
        "frames": {"sampled": c["sampled_frames"], "candidates": cand_total, "empty": neg_total, "audit_sampled": c["audit_frames"]},
        "review_coverage": {"candidates_decided": tp + fp, "candidates_flagged": fl_c,
                            "candidates_pending": max(cand_total - (tp + fp + fl_c), 0),
                            "audit_decided": fn_a + tn_a, "audit_flagged": fl_a},
        "counts": {"TP": tp, "FP": fp, "flagged_candidates": fl_c, "audit_person_present(FN)": fn_a, "audit_empty(TN)": tn_a},
        "estimated_counts": {"TP": tp_e, "FP": fp_e, "FN": fn_e, "TN": tn_e},
        "precision": precision, "precision_ci95": [p_lo, p_hi],
        "precision_if_flags_are_FP": _ratio(tp, tp + fp + fl_c), "precision_if_flags_are_TP": _ratio(tp + fl_c, tp + fp + fl_c),
        "recall_estimate": recall, "false_positive_rate_estimate": fpr, "f1_estimate": f1,
        "audit_miss_rate_ci95": [miss_lo, miss_hi],
        "speed": {
            "seconds_per_frame_inference": t["inference_s"] / n, "fps_inference": n / max(t["inference_s"], 1e-9),
            "seconds_per_frame_end_to_end": t["wall_s"] / n, "fps_end_to_end": n / max(t["wall_s"], 1e-9),
            "decode_seconds_per_frame": t["decode_s"] / n, "p95_inference_s": t["p95_infer_s"],
            "realtime_factor": t["source_seconds_covered"] / max(t["wall_s"], 1e-9), "device": summary["device"],
        },
    }


def threshold_sweep(log_df: pd.DataFrame) -> pd.DataFrame:
    """What if CFG.conf had been higher? Uses reviewed candidate frames (max confidence per frame)."""
    d = log_df[(log_df["frame_kind"] == "candidate") & log_df["decision"].isin([DECISION_CONFIRMED, DECISION_REJECTED])].copy()
    if d.empty:
        return pd.DataFrame(columns=["threshold", "precision", "recall_vs_confirmed", "f1_vs_confirmed", "frames_kept"])
    d["conf"] = d["max_confidence"].astype(float)
    pos = (d["decision"] == DECISION_CONFIRMED)
    total_pos = int(pos.sum())
    rows = []
    for th in np.linspace(max(0.05, d["conf"].min()), min(0.95, d["conf"].max() + 1e-6), 25):
        kept = d["conf"] >= th
        tp, fp = int((kept & pos).sum()), int((kept & ~pos).sum())
        fn = total_pos - tp
        rows.append({"threshold": float(th), "precision": _ratio(tp, tp + fp), "recall_vs_confirmed": _ratio(tp, total_pos),
                     "f1_vs_confirmed": _ratio(2 * tp, 2 * tp + fp + fn), "frames_kept": int(kept.sum())})
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------
# B) Ground-truth evaluation (labelled datasets)
# ----------------------------------------------------------------------------
def _iou_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.maximum(x2 - x1, 0) * np.maximum(y2 - y1, 0)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / np.maximum(area_a[:, None] + area_b[None, :] - inter, 1e-9)


def match_detections(dets: np.ndarray, gts: np.ndarray, iou_thr: float) -> Tuple[int, int, int]:
    """Greedy one-to-one matching by confidence. Returns (TP, FP, FN) for one frame."""
    if len(dets) == 0:
        return 0, 0, len(gts)
    order = np.argsort(-dets[:, 4])
    iou = _iou_matrix(dets[order, :4], gts)
    used, tp = set(), 0
    for i in range(len(order)):
        if iou.shape[1] == 0:
            break
        j = int(np.argmax(np.where([k in used for k in range(iou.shape[1])], -1.0, iou[i])))
        if j not in used and iou[i, j] >= iou_thr:
            used.add(j)
            tp += 1
    return tp, len(dets) - tp, len(gts) - tp


def compute_ground_truth_metrics(all_frames: List[dict], gt: Dict[str, np.ndarray], iou_thr: float = 0.5,
                                 min_detections: int = 1) -> dict:
    TP = FP = FN = 0
    fTP = fFP = fFN = fTN = 0
    per_frame_truth: Dict[str, bool] = {}
    for fr in all_frames:
        g = gt.get(fr["frame_id"])
        if g is None:
            continue
        dets = np.asarray(fr["detections"], dtype=np.float32).reshape(-1, 5)
        tp, fp, fn = match_detections(dets, g, iou_thr)
        TP, FP, FN = TP + tp, FP + fp, FN + fn
        truth, pred = len(g) > 0, len(dets) >= min_detections
        per_frame_truth[fr["frame_id"]] = truth
        fTP += int(truth and pred); fFP += int(pred and not truth)
        fFN += int(truth and not pred); fTN += int(not truth and not pred)
    prec, rec = _ratio(TP, TP + FP), _ratio(TP, TP + FN)
    return {
        "frames_with_labels": len(per_frame_truth), "frames_without_labels": len(all_frames) - len(per_frame_truth),
        "iou_threshold": iou_thr,
        "detection_level": {"TP": TP, "FP": FP, "FN": FN, "precision": prec, "recall": rec,
                            "f1": _ratio(2 * prec * rec, prec + rec) if (prec and rec) else None},
        "frame_level": {"TP": fTP, "FP": fFP, "FN": fFN, "TN": fTN, "precision": _ratio(fTP, fTP + fFP),
                        "recall": _ratio(fTP, fTP + fFN), "false_positive_rate": _ratio(fFP, fFP + fTN)},
        "_per_frame_truth": per_frame_truth,
    }


def reviewer_vs_ground_truth(log_df: pd.DataFrame, per_frame_truth: Dict[str, bool]) -> dict:
    """How often did the human decision match the labelled truth? (flagged frames excluded)"""
    d = log_df[log_df["decision"].isin([DECISION_CONFIRMED, DECISION_REJECTED]) & log_df["frame_id"].isin(per_frame_truth)]
    if d.empty:
        return {"reviewed_with_gt": 0, "agreement": None}
    agree = sum((row.decision == DECISION_CONFIRMED) == per_frame_truth[row.frame_id] for row in d.itertuples())
    return {"reviewed_with_gt": int(len(d)), "agreement": agree / len(d)}


# ----------------------------------------------------------------------------
# Reporting
# ----------------------------------------------------------------------------
def _fmt(v, pct=False, nd=3):
    if v is None:
        return "n/a (needs audit review)"
    return f"{v * 100:.1f} %" if pct else f"{v:.{nd}f}"


def run_evaluation(audit_trail: "AuditTrail", result: "RunResult", cfg, gt_iou: float = 0.5) -> dict:
    log_df = audit_trail.to_dataframe()
    m = compute_review_metrics(log_df, result.summary)
    sweep = threshold_sweep(log_df)
    report = {"run_id": result.run_id, "data_source": result.summary.get("data_source_label"), "reviewer_based": m}

    rows = [
        ("Precision (reviewed candidates)", _fmt(m["precision"], True)),
        ("   95 % CI", "n/a" if m["precision_ci95"][0] is None else f"{m['precision_ci95'][0]*100:.1f} – {m['precision_ci95'][1]*100:.1f} %"),
        ("   if flagged = FP / = TP", f"{_fmt(m['precision_if_flags_are_FP'], True)} / {_fmt(m['precision_if_flags_are_TP'], True)}"),
        ("Recall (estimated from audit)", _fmt(m["recall_estimate"], True)),
        ("False-positive rate (est.)", _fmt(m["false_positive_rate_estimate"], True)),
        ("F1 (estimated)", _fmt(m["f1_estimate"], nd=3)),
        ("Seconds / frame (inference)", f"{m['speed']['seconds_per_frame_inference']:.3f} s"),
        ("FPS (inference only)", f"{m['speed']['fps_inference']:.2f}"),
        ("Seconds / frame (end-to-end)", f"{m['speed']['seconds_per_frame_end_to_end']:.3f} s"),
        ("FPS (end-to-end)", f"{m['speed']['fps_end_to_end']:.2f}"),
        ("Real-time factor (source s / wall s)", f"{m['speed']['realtime_factor']:.2f}×" if result.summary.get("source_mode") == "video" else "n/a (image folder: no video clock)"),
        ("Device", m["speed"]["device"].upper()),
    ]
    print("═" * 70)
    print(f" AAROHAN evaluation — run {result.run_id}\n source: {result.summary.get('data_source_label')}")
    print("═" * 70)
    display(pd.DataFrame(rows, columns=["Metric", "Value"]).style.hide(axis="index"))
    cov = m["review_coverage"]
    print(f"Review coverage: {cov['candidates_decided']} decided + {cov['candidates_flagged']} flagged + "
          f"{cov['candidates_pending']} pending candidate frames; {cov['audit_decided']} audit frames decided.")
    if m["recall_estimate"] is None:
        print("ℹ Recall / FPR need reviewed AUDIT frames (set CFG.audit_fraction > 0 and review them).")

    gt_res = None
    if cfg.labels_dir is not None and result.all_frames:
        gt = load_ground_truth(result.all_frames, cfg.labels_dir)
        if gt:
            gt_res = compute_ground_truth_metrics(result.all_frames, gt, gt_iou, cfg.min_detections)
            truth = gt_res.pop("_per_frame_truth")
            gt_res["reviewer_vs_ground_truth"] = reviewer_vs_ground_truth(log_df, truth)
            dl, fl = gt_res["detection_level"], gt_res["frame_level"]
            print("\n── Ground-truth evaluation (labelled dataset) ──")
            display(pd.DataFrame([
                ["Detection-level precision / recall / F1 @IoU≥%.2f" % gt_iou, f"{_fmt(dl['precision'], True)} / {_fmt(dl['recall'], True)} / {_fmt(dl['f1'])}"],
                ["Frame-level precision / recall", f"{_fmt(fl['precision'], True)} / {_fmt(fl['recall'], True)}"],
                ["Frame-level false-positive rate", _fmt(fl["false_positive_rate"], True)],
                ["Frames with / without labels", f"{gt_res['frames_with_labels']} / {gt_res['frames_without_labels']}"],
                ["Reviewer agreement with ground truth", _fmt(gt_res["reviewer_vs_ground_truth"]["agreement"], True)],
            ], columns=["Metric", "Value"]).style.hide(axis="index"))
        else:
            print("ℹ CFG.labels_dir is set but no label files matched the image names.")
    report["ground_truth"] = gt_res

    # ---- figure ----
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
    est = m["estimated_counts"]
    labels = ["TP", "FP", "FN (est.)", "TN (est.)"]
    vals = [est["TP"], est["FP"], est["FN"] or 0, est["TN"] or 0]
    ax[0].bar(labels, vals, color=["#1a7f37", "#cf222e", "#8250df", "#6e7781"])
    ax[0].set_title("Frame-level outcome (audit-scaled estimates)"); ax[0].set_ylabel("frames")
    cd = log_df[(log_df["frame_kind"] == "candidate") & log_df["decision"].isin([DECISION_CONFIRMED, DECISION_REJECTED])]
    if len(cd):
        bins = np.linspace(0, 1, 21)
        ax[1].hist(cd[cd["decision"] == DECISION_CONFIRMED]["max_confidence"].astype(float), bins=bins, alpha=.75, color="#1a7f37", label="confirmed")
        ax[1].hist(cd[cd["decision"] == DECISION_REJECTED]["max_confidence"].astype(float), bins=bins, alpha=.75, color="#cf222e", label="rejected")
        ax[1].legend()
    ax[1].set_title("Max confidence per reviewed frame"); ax[1].set_xlabel("confidence"); ax[1].set_ylabel("frames")
    if len(sweep):
        ax[2].plot(sweep["threshold"], sweep["precision"], label="precision", color="#0969da")
        ax[2].plot(sweep["threshold"], sweep["recall_vs_confirmed"], label="recall vs confirmed", color="#bf8700")
        ax[2].plot(sweep["threshold"], sweep["f1_vs_confirmed"], label="F1", color="#1a7f37", ls="--")
        n_pos = int((cd["decision"] == DECISION_CONFIRMED).sum())
        n_neg_ = int((cd["decision"] == DECISION_REJECTED).sum())
        enough = (n_pos + n_neg_) >= 30 and n_pos >= 5 and n_neg_ >= 5      # guard against advice from a handful of frames
        ax[2].legend(loc="lower left")
        if enough:
            best = sweep.loc[sweep["f1_vs_confirmed"].astype(float).idxmax()]
            ax[2].axvline(best["threshold"], color="grey", ls=":")
            report["suggested_confidence_threshold"] = float(best["threshold"])
            print(f"\nSuggested CFG.conf from reviewed frames (max-F1 on confirmed set): {best['threshold']:.2f}  "
                  f"(current: {cfg.conf}). Re-run Cell 3 to apply — recall for unreviewed misses is not captured here.")
        else:
            print(f"\nℹ No confidence-threshold suggestion: only {n_pos} confirmed / {n_neg_} rejected candidate frames reviewed "
                  f"(need >= 30 reviewed with at least 5 of each). Review more frames first.")
    ax[2].set_title("What-if confidence threshold"); ax[2].set_xlabel("threshold"); ax[2].set_ylim(0, 1.02)
    plt.tight_layout(); plt.show()

    out = Path(result.run_dir) / "evaluation_report.json"
    out.write_text(json.dumps(report, indent=2, default=str))
    print(f"✔ Saved {out}")
    return report


# ----------------------------------------------------------------------------
# C) Ground-truth ONLY (no reviewer needed): objective benchmark on a labelled dataset
# ----------------------------------------------------------------------------
def ground_truth_sweep(all_frames: List[dict], gt: Dict[str, np.ndarray], thresholds, iou_thr: float = 0.5) -> pd.DataFrame:
    """Detection-level precision / recall / F1 at several confidence thresholds (thresholds must be >= CFG.conf,
    because detections below CFG.conf were never stored)."""
    rows = []
    for t in thresholds:
        TP = FP = FN = 0
        for fr in all_frames:
            g = gt.get(fr["frame_id"])
            if g is None:
                continue
            dets = np.asarray(fr["detections"], dtype=np.float32).reshape(-1, 5)
            dets = dets[dets[:, 4] >= t]
            tp, fp, fn = match_detections(dets, g, iou_thr)
            TP, FP, FN = TP + tp, FP + fp, FN + fn
        p, r = _ratio(TP, TP + FP), _ratio(TP, TP + FN)
        rows.append({"threshold": float(t), "TP": TP, "FP": FP, "FN": FN, "precision": p, "recall": r,
                     "f1": _ratio(2 * TP, 2 * TP + FP + FN)})
    return pd.DataFrame(rows)


def run_ground_truth_only(result: "RunResult", cfg, gt_iou: float = 0.5) -> Optional[dict]:
    gt = load_ground_truth(result.all_frames, cfg.labels_dir)
    if not gt:
        print("ℹ No label files matched the image names — check CFG.labels_dir.")
        return None
    base = compute_ground_truth_metrics(result.all_frames, gt, gt_iou, cfg.min_detections)
    base.pop("_per_frame_truth")
    ths = np.unique(np.round(np.arange(max(cfg.conf, 0.05), 0.96, 0.05), 3))
    sweep = ground_truth_sweep(result.all_frames, gt, ths, gt_iou)
    dl, fl = base["detection_level"], base["frame_level"]
    c, t = result.summary["counts"], result.summary["timing"]
    n = max(c["sampled_frames"], 1)
    print("═" * 70)
    print(f" AAROHAN ground-truth benchmark — run {result.run_id}\n source: {result.summary.get('data_source_label')}")
    print(f" {base['frames_with_labels']} labelled images ({base['frames_without_labels']} without labels) | IoU >= {gt_iou}")
    print("═" * 70)
    display(pd.DataFrame([
        [f"Detection-level precision (conf >= {cfg.conf})", _fmt(dl["precision"], True)],
        ["Detection-level recall", _fmt(dl["recall"], True)],
        ["Detection-level F1", _fmt(dl["f1"])],
        ["TP / FP / FN (boxes)", f"{dl['TP']} / {dl['FP']} / {dl['FN']}"],
        ["Frame-level precision / recall", f"{_fmt(fl['precision'], True)} / {_fmt(fl['recall'], True)}"],
        ["Frame-level false-positive rate", _fmt(fl["false_positive_rate"], True) if (fl["FP"] + fl["TN"]) > 0
         else "n/a (no empty frames in this labelled set)"],
        ["Seconds / frame (inference)", f"{t['inference_s'] / n:.3f} s"],
        ["FPS (inference only / end-to-end)", f"{n / max(t['inference_s'], 1e-9):.2f} / {n / max(t['wall_s'], 1e-9):.2f}"],
        ["Device", result.summary["device"].upper()],
    ], columns=["Metric", "Value"]).style.hide(axis="index"))
    if (fl["FP"] + fl["TN"]) == 0:
        print("ℹ Every labelled frame contains people, so frame-level precision / FPR are not informative here "
              "(100 % precision is automatic). Use the detection-level numbers.")
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
    ax[0].plot(sweep["threshold"], sweep["precision"], label="precision", color="#0969da")
    ax[0].plot(sweep["threshold"], sweep["recall"], label="recall", color="#bf8700")
    ax[0].plot(sweep["threshold"], sweep["f1"], label="F1", color="#1a7f37", ls="--")
    ax[0].set_xlabel("confidence threshold"); ax[0].set_ylim(0, 1.02); ax[0].legend(); ax[0].set_title("Objective P / R / F1 vs threshold")
    ax[1].plot(sweep["recall"], sweep["precision"], marker="o", color="#8250df")
    ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision"); ax[1].set_xlim(0, 1.02); ax[1].set_ylim(0, 1.02)
    ax[1].set_title("Precision–recall curve (detection level)")
    report = {"run_id": result.run_id, "data_source": result.summary.get("data_source_label"), "ground_truth": base,
              "sweep": sweep.to_dict(orient="records")}
    enough = base["frames_with_labels"] >= 30
    if enough and sweep["f1"].notna().any():
        best = sweep.loc[sweep["f1"].astype(float).idxmax()]
        ax[0].axvline(best["threshold"], color="grey", ls=":")
        report["best_f1_threshold"] = float(best["threshold"])
        print(f"\nBest F1 on this labelled set at conf >= {best['threshold']:.2f}  (P {_fmt(best['precision'], True)}, "
              f"R {_fmt(best['recall'], True)}). To test thresholds BELOW {cfg.conf}, lower CFG.conf and re-run Cell 3.")
    else:
        print(f"\nℹ Only {base['frames_with_labels']} labelled images — too few for a threshold recommendation (need >= 30).")
    plt.tight_layout(); plt.show()
    out = Path(result.run_dir) / "evaluation_gt_report.json"
    out.write_text(json.dumps(report, indent=2, default=str))
    print(f"✔ Saved {out}")
    return report


_res, _audit = globals().get("RESULT"), globals().get("audit")
METRICS = None
if _res is None:
    print("ℹ Run Cells 3–5 first, then re-run this cell.")
elif _audit is None or len(_audit._state) == 0:
    if CFG.labels_dir is not None and _res.all_frames:
        print("ℹ No reviewer decisions yet → running the GROUND-TRUTH benchmark (no review needed).")
        METRICS = run_ground_truth_only(_res, CFG)
    else:
        print("ℹ No reviewer decisions recorded yet — review frames in Cell 4, then re-run this cell.")
else:
    METRICS = run_evaluation(_audit, _res, CFG)


---
## Cell 7 — Fine-tune YOLO on HERIDAL and compare with baseline

Opt-in, **three stages — enable one flag at a time**, run the cell, then the next:

| Flag | Stage |
|------|-------|
| `RUN_FT_DOWNLOAD = True` | Random train/val subset (test images never used) |
| `RUN_FT_TILING = True` | 4000×3000 → 640-px tiles + YOLO labels + hard negatives |
| `RUN_FT_TRAIN = True` | Train on MPS; copy `best.pt` → `~/AAROHAN/models/` |

Then:
```python
evaluate_weights(FT_WEIGHTS)   # same 101 test images
compare_gt_reports()           # before/after table
```

Honest scope: gains on HERIDAL do not prove performance on other terrain.


In [ ]:
# ==============================================================================
# AAROHAN | CELL 7 — Fine-tune YOLO on HERIDAL (aerial persons) and compare with the baseline
# ==============================================================================
# Why: the COCO-pretrained baseline misses most tiny aerial persons (see Cell 6 benchmark).
# Plan (run the three stages one after another, each behind its own flag):
#   Stage 1  RUN_FT_DOWNLOAD : fetch a RANDOM subset of HERIDAL train + val images and labels (not the test split)
#   Stage 2  RUN_FT_TILING   : cut every 4000x3000 image into 640x640 tiles + YOLO labels (+ a few empty tiles as negatives)
#   Stage 3  RUN_FT_TRAIN    : train on the Mac GPU (MPS), copy best.pt to ~/AAROHAN/models/
# Then:  evaluate_weights(<best.pt>)  re-runs the SAME 101 test images → compare_gt_reports() prints the before/after table.
# Needs Cells 1, 2, 2C and 3 to have been run (it reuses fetch_subset, tile_origins, load_model, process_source, ...).
# Honest scope: train/val/test come from the same dataset authors' split. Test images are NEVER used for training or for
# choosing best.pt (val does that). Gains on HERIDAL do not prove performance on other terrain.
# ------------------------------------------------------------------------------
# ---------------- USER SETTINGS ----------------
import shutil

FT_N_TRAIN = 150            # random train images to download (≈ 4.9 MB each; 150 ≈ 0.75 GB)
FT_N_VAL = 40               # random val images (used for early stopping / best.pt choice)
FT_SEED = 42
FT_TILE = 640               # tile size in pixels (matches CFG.tile_size for consistent train/test scale)
FT_OVERLAP = 0.20           # tile overlap → stride 512
FT_NEG_RATIO = 0.15         # empty tiles kept per positive tile (hard negatives reduce false alarms)
FT_MIN_VISIBLE = 0.5        # keep a box in a tile only if >= 50 % of it is inside; tiles with 5–50 % slivers are skipped
FT_BASE_WEIGHTS = "yolo11s.pt"
FT_EPOCHS = 30
FT_BATCH = 16               # lower to 8 if MPS runs out of memory
FT_PATIENCE = 10            # early stopping on val
FT_RESUME = False           # True → continue an interrupted training from last.pt
RUN_FT_DOWNLOAD = False     # Stage 1
RUN_FT_TILING = False       # Stage 2
RUN_FT_TRAIN = False        # Stage 3

FT_ROOT = CFG.project_dir / "datasets" / "HERIDAL_ft"
FT_RAW = FT_ROOT / "raw"                      # downloaded full-size images + XML
FT_YOLO = FT_ROOT / "yolo"                    # tiles + YOLO labels + dataset.yaml
FT_RUNS = CFG.project_dir / "finetune"
FT_MODELS = CFG.project_dir / "models"


# ----------------------------------------------------------------------------
# Stage 1 — choose a leak-free random subset and download it
# ----------------------------------------------------------------------------
def ft_select_ids(zf, n_train: int, n_val: int, seed: int):
    """Random train/val ids that HAVE an XML label. Raises if any id overlaps the test split (leakage guard)."""
    info = {i.filename for i in zf.infolist()}
    def ids_of(name):
        return zf.read(HERIDAL_BASE + f"ImageSets/Main/{name}.txt").decode().split()
    test_ids = set(ids_of("test"))
    pools = {}
    for split in ("train", "val"):
        pool = [i for i in ids_of(split) if HERIDAL_BASE + f"Annotations/{i}.xml" in info and i not in test_ids]
        pools[split] = sorted(set(pool))
    overlap = set(pools["train"]) & set(pools["val"])
    pools["train"] = [i for i in pools["train"] if i not in overlap]          # val wins on overlap
    rng = random.Random(seed)
    train = sorted(rng.sample(pools["train"], min(n_train, len(pools["train"]))))
    val = sorted(rng.sample(pools["val"], min(n_val, len(pools["val"]))))
    assert not (set(train) | set(val)) & test_ids, "LEAKAGE: test image selected for training"
    return train, val, len(pools["train"]), len(pools["val"])


def ft_download(opener=None) -> dict:
    if opener is None:
        from remotezip import RemoteZip
        opener = lambda: RemoteZip(HERIDAL_URL)                                 # noqa: E731
    out = {}
    with opener() as z:
        train, val, ptr, pva = ft_select_ids(z, FT_N_TRAIN, FT_N_VAL, FT_SEED)
        print(f"Selected {len(train)} train (of {ptr} labelled) + {len(val)} val (of {pva}) | seed={FT_SEED}")
        for split, ids in (("train", train), ("val", val)):
            done, missing = fetch_subset(z, ids, FT_RAW / split / "images", FT_RAW / split / "labels", HERIDAL_BASE)
            out[split] = ids
            print(f"  {split}: {done} files ready, missing: {missing}")
    (FT_ROOT / "selection.json").write_text(json.dumps({"seed": FT_SEED, **out}, indent=1))
    return out


# ----------------------------------------------------------------------------
# Stage 2 — full image → tiles + YOLO labels
# ----------------------------------------------------------------------------
def tile_one_image(img: np.ndarray, boxes: np.ndarray, rng: random.Random):
    """Yield (x, y, tile_bgr, yolo_lines, is_positive) for every usable tile of one image."""
    H, W = img.shape[:2]
    for (x, y, tw, th) in tile_origins(W, H, FT_TILE, FT_OVERLAP):
        lines, ambiguous = [], False
        for (bx1, by1, bx2, by2) in boxes:
            ix1, iy1, ix2, iy2 = max(bx1, x), max(by1, y), min(bx2, x + tw), min(by2, y + th)
            inter = max(ix2 - ix1, 0) * max(iy2 - iy1, 0)
            area = max((bx2 - bx1) * (by2 - by1), 1e-9)
            frac = inter / area
            if frac >= FT_MIN_VISIBLE:
                cx, cy = ((ix1 + ix2) / 2 - x) / tw, ((iy1 + iy2) / 2 - y) / th
                lines.append(f"0 {cx:.6f} {cy:.6f} {(ix2 - ix1) / tw:.6f} {(iy2 - iy1) / th:.6f}")
            elif frac > 0.05:
                ambiguous = True                                              # a person sliver we cannot label cleanly
        if ambiguous:
            continue
        yield x, y, img[y:y + th, x:x + tw], lines, bool(lines)


def ft_tiling() -> dict:
    stats = {}
    for split in ("train", "val"):
        img_in, lbl_in = FT_RAW / split / "images", FT_RAW / split / "labels"
        img_out, lbl_out = FT_YOLO / "images" / split, FT_YOLO / "labels" / split
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)
        marker = FT_YOLO / f".done_{split}.txt"
        done = set(marker.read_text().split()) if marker.exists() else set()
        rng = random.Random(FT_SEED + (0 if split == "train" else 1))
        pos = neg = boxes_n = 0
        files = sorted(img_in.glob("*"))
        for p in tqdm(files, desc=f"tiling {split}", unit="img"):
            if p.stem in done:
                continue
            img = load_frame(p)
            xml = lbl_in / f"{p.stem}.xml"
            if img is None or not xml.exists():
                log.warning("skip %s (unreadable image or missing label)", p.name)
                continue
            boxes = _read_voc_labels(xml)
            tiles = list(tile_one_image(img, boxes, rng))
            positives = [t for t in tiles if t[4]]
            negatives = [t for t in tiles if not t[4]]
            keep_neg = rng.sample(negatives, min(len(negatives), int(round(FT_NEG_RATIO * max(len(positives), 1)))))
            for (x, y, tile, lines, is_pos) in positives + keep_neg:
                name = f"{p.stem}_x{x}_y{y}"
                save_frame_jpeg(tile, img_out / f"{name}.jpg", 95)
                (lbl_out / f"{name}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
            pos += len(positives); neg += len(keep_neg); boxes_n += len(boxes)
            with open(marker, "a") as fh:
                fh.write(p.stem + "\n")
            del img, tiles
        stats[split] = {"positive_tiles": pos, "negative_tiles": neg, "gt_boxes": boxes_n}
    (FT_YOLO / "dataset.yaml").write_text(
        f"path: {FT_YOLO}\ntrain: images/train\nval: images/val\nnames:\n  0: person\n")
    print("Tiling done (this run):", stats)
    print("Tiles on disk:", {s: len(list((FT_YOLO / 'images' / s).glob('*.jpg'))) for s in ('train', 'val')},
          "→", FT_YOLO / "dataset.yaml")
    return stats


# ----------------------------------------------------------------------------
# Stage 3 — train
# ----------------------------------------------------------------------------
def ft_train(epochs: int = None, base_weights: str = None, fraction: float = 1.0, imgsz: int = None) -> Path:
    epochs = epochs or FT_EPOCHS
    base_weights = base_weights or FT_BASE_WEIGHTS
    imgsz = imgsz or FT_TILE
    yaml_path = FT_YOLO / "dataset.yaml"
    if not yaml_path.exists():
        raise FileNotFoundError("Run Stage 2 (tiling) first — dataset.yaml not found.")
    FT_RUNS.mkdir(parents=True, exist_ok=True)
    run_name = f"heridal_{Path(base_weights).stem}"
    last = FT_RUNS / run_name / "weights" / "last.pt"
    if FT_RESUME and last.exists():
        model = YOLO(str(last))
        model.train(resume=True)
    else:
        model = YOLO(base_weights)
        model.train(
            data=str(yaml_path), imgsz=imgsz, epochs=epochs, batch=FT_BATCH, device=CFG.device,
            workers=0,                       # macOS + Jupyter: worker processes can hang
            patience=FT_PATIENCE, plots=False, project=str(FT_RUNS), name=run_name, exist_ok=True,
            flipud=0.5, fliplr=0.5,          # top-down views have no 'up' → vertical flips are valid augmentation
            fraction=fraction, seed=FT_SEED, verbose=True,
        )
    best = FT_RUNS / run_name / "weights" / "best.pt"
    if not best.exists():
        best = last
    FT_MODELS.mkdir(parents=True, exist_ok=True)
    final = FT_MODELS / f"aarohan_{run_name}.pt"
    shutil.copy(best, final)
    print(f"✔ Fine-tuned weights → {final}")
    return final


# ----------------------------------------------------------------------------
# Evaluate on the SAME held-out test images + before/after table
# ----------------------------------------------------------------------------
def evaluate_weights(weights, conf: float = 0.10, gt_iou: float = 0.5, label: str = None):
    """New run on the HERIDAL test split with the given weights; prints the ground-truth benchmark."""
    global RESULT, MODEL
    img_dir = CFG.project_dir / "datasets" / "HERIDAL_test" / "images"
    lbl_dir = CFG.project_dir / "datasets" / "HERIDAL_test" / "labels"
    if not (img_dir.exists() and lbl_dir.exists()):
        raise FileNotFoundError("HERIDAL test split not found — run Cell 2C first.")
    CFG.source_mode, CFG.images_dir, CFG.labels_dir, CFG.max_frames = "images", img_dir, lbl_dir, None
    CFG.weights, CFG.conf, CFG.audit_fraction = str(weights), conf, 0.0
    CFG.data_source_label = label or f"HERIDAL test split ({len(list(img_dir.glob('*')))} images) | weights={Path(str(weights)).name}"
    CFG.run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    CFG.run_dir.mkdir(parents=True, exist_ok=True)
    CFG.frames_dir.mkdir(parents=True, exist_ok=True)
    MODEL = load_model(CFG)
    RESULT = process_source(CFG, MODEL)
    print_run_summary(RESULT)
    return run_ground_truth_only(RESULT, CFG, gt_iou)


def compare_gt_reports() -> pd.DataFrame:
    """One row per finished ground-truth benchmark run (baseline vs fine-tuned vs experiments)."""
    rows = []
    for rep in sorted((CFG.project_dir / "runs").glob("*/evaluation_gt_report.json")):
        try:
            r = json.loads(rep.read_text())
            s = json.loads((rep.parent / "run_summary.json").read_text())
            dl = r["ground_truth"]["detection_level"]
            n = max(s["counts"]["sampled_frames"], 1)
            rows.append({"run_id": r["run_id"], "weights": Path(s["weights"]).name, "conf": s["config"]["conf"],
                         "tile_imgsz": s["config"]["tile_imgsz"], "images": r["ground_truth"]["frames_with_labels"],
                         "IoU": r["ground_truth"]["iou_threshold"],
                         "precision": None if dl["precision"] is None else round(dl["precision"], 3),
                         "recall": None if dl["recall"] is None else round(dl["recall"], 3),
                         "F1": None if dl["f1"] is None else round(dl["f1"], 3),
                         "s/frame": round(s["timing"]["inference_s"] / n, 3)})
        except Exception as exc:
            log.warning("skipped %s (%s)", rep, exc)
    df = pd.DataFrame(rows)
    if len(df):
        display(df.style.hide(axis="index"))
    else:
        print("No ground-truth benchmark runs found yet (run Cell 6 on labelled data first).")
    return df


# ----------------------------------------------------------------------------
# Run the stages you enabled
# ----------------------------------------------------------------------------
if not any((RUN_FT_DOWNLOAD, RUN_FT_TILING, RUN_FT_TRAIN)):
    print("ℹ Cell 7 is idle. Enable ONE stage at a time (RUN_FT_DOWNLOAD → RUN_FT_TILING → RUN_FT_TRAIN) and re-run this cell.")
if RUN_FT_DOWNLOAD:
    FT_SELECTION = ft_download()
if RUN_FT_TILING:
    FT_STATS = ft_tiling()
if RUN_FT_TRAIN:
    FT_WEIGHTS = ft_train()
    print("Next: evaluate_weights(FT_WEIGHTS) and compare_gt_reports()")


---
## Notes & limitations (read before quoting numbers)

* Public annotated SAR datasets use volunteers/actors on real terrain; real casualty imagery is not published. Quote Track A numbers as *benchmark* performance and Track B as *field-test* behaviour.
* Default weights are COCO-trained (ground-level photos). Expect weaker recall on tiny aerial persons than on COCO; fine-tune on HERIDAL / SARD before claiming operational performance.
* Estimated recall / FPR depend on the audit sample size; the confidence interval is printed for precision, and the audit miss-rate interval is stored in `evaluation_report.json`.
* Output folder per run: `~/AAROHAN/runs/<run_id>/` (frames, `candidates.json`, `all_frames.json`, `run_summary.json`, `detections_log.csv`, `review_events.csv`, `evaluation_report.json`).
* Safety: the tool supports human review; it does **not** autonomously decide who needs rescue or direct a drone.
